# recursive_opt — Use-Case Experiment Suite

This notebook runs **3 complementary experiments per use case** to maximize the chance
of a successful / informative result, measures them in a comparison table, and
displays the winning artifact/code so you can inspect and reuse it.

## Read this first: live vs offline preflight mode
A Trace **optimizer** (OptoPrimeV2) calls an LLM, so genuine recursive optimization
requires `LIVE=True` and an API key. This notebook is intended to be run live for
evidence. If you explicitly set `RECURSIVE_OPT_LIVE=0`, it only writes inspectable
specs and runs explicitly marked offline preflights. Those rows must not be
interpreted as live optimization evidence.


In [1]:
# ============================ CONFIGURATION ===============================
# LIVE=True runs REAL recursive optimization. The model is configured here so
# every optimizer / Trace-Bench inference path uses the same backend.
import os, sys, json, time, statistics, textwrap, math
from pathlib import Path

# Make Run-All robust from repo root, examples/, or nbconvert kernels whose cwd
# is not automatically inserted on sys.path. This is notebook-only path setup; it
# does not change the installed package.
_REPO_ROOT = Path.cwd()
if not (_REPO_ROOT / "opto").exists() and (_REPO_ROOT.parent / "opto").exists():
    _REPO_ROOT = _REPO_ROOT.parent
if str(_REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(_REPO_ROOT))

LIVE = os.environ.get("RECURSIVE_OPT_LIVE", "true").strip().lower() not in {"0", "false", "off", "no"}
if LIVE and not any(os.environ.get(k) for k in ("OPENAI_API_KEY", "OPENROUTER_API_KEY", "OPENAI_ADMIN_KEY")):
    raise RuntimeError(
        "LIVE recursive_opt run requested but no API key is set. "
        "Set OPENAI_API_KEY / OPENROUTER_API_KEY, or explicitly set RECURSIVE_OPT_LIVE=0 for offline spec/preflight inspection."
    )
MODEL = os.environ.get("RECURSIVE_OPT_MODEL") or os.environ.get("TRACE_LITELLM_MODEL") or "gpt-5.4-nano"
os.environ["RECURSIVE_OPT_MODEL"] = MODEL
os.environ["TRACE_LITELLM_MODEL"] = MODEL

# Budget - these map 1:1 to spec["budget"] (RecursiveOptBudget). Tune per run.
WALL_TIME_S          = 1800  # hard wall-clock cap per run (~30 min); the simplest guard
RUN_ITERATIONS       = 2     # optimizer update steps per seed/run
NUM_CANDIDATES       = 2     # proposals per optimizer step
MAX_OPTIMIZER_CALLS  = 8     # caps LLM proposal cost per seed/run
MAX_EVAL_CALLS       = 48    # standard cap for prompt/config/code runs
CAPABILITY_EVAL_CALLS = 96  # UC3 does train + final eval over 8 examples; 48 exhausted before save_priors
MAX_CANDIDATES       = 8     # caps search breadth (iterations * num_candidates)
os.environ["RECURSIVE_OPT_ITERATIONS"] = str(RUN_ITERATIONS)
os.environ["RECURSIVE_OPT_NUM_CANDIDATES"] = str(NUM_CANDIDATES)

# Task-eval bounds dominate cost more than anything. 8 examples is the current
# default here because earlier 4-example runs saturated too easily and were noisy.
MAX_EXAMPLES = 8
HARD_MAX_EXAMPLES = int(os.environ.get("RECURSIVE_OPT_HARD_MAX_EXAMPLES", "4"))  # HF QA tasks are slower; 4 reduces single-example noise without making Run-All impractical
INNER_STEPS  = 0             # 0 = artifact-only (fast); >0 = real inner training (costly)
TIMEOUT_S    = 35            # per-eval timeout
os.environ["RECURSIVE_OPT_CAPABILITY_MAX_EXAMPLES"] = str(MAX_EXAMPLES)

# Five seeds are the minimum useful live check for noisy frontier/code-policy arms.
# Earlier n=1 diagnostic runs produced unstable verdict flips, so three-way cells
# intentionally share the same five-seed set unless a cell overrides it explicitly.
SEEDS = [0, 1, 2, 3, 4]
DIAGNOSTIC_SEEDS = SEEDS

# Keep every generated memory/artifact folder under one run directory.
# If the notebook is launched from the repo root, this resolves to
# examples/notebook_outputs/...; if launched from examples/, it resolves to
# notebook_outputs/... under repo examples/. Override with RECURSIVE_OPT_OUTPUT_ROOT.
RUN_ID = os.environ.get("RECURSIVE_OPT_RUN_ID") or time.strftime("use_cases_%Y%m%d_%H%M%S")
_DEFAULT_OUTPUT_ROOT = _REPO_ROOT / "examples/notebook_outputs/recursive_opt_use_cases"
OUTPUT_ROOT = Path(os.environ.get("RECURSIVE_OPT_OUTPUT_ROOT", str(_DEFAULT_OUTPUT_ROOT))) / RUN_ID
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

# credit_horizon controls how much per-example guide feedback the optimizer sees.
# For UC6 we fix it at the previously best setting ("step") and compare trace designs.

# NOTE on the hf:gsm8k alias: recursive_opt currently redirects hf:gsm8k ->
# internal:multiobjective_gsm8k. We use internal:* families directly to avoid ambiguity.
if LIVE:
    from opto.features.recursive_opt.runmode import preflight_model
    from opto.features.recursive_opt.tracebench import ensure_default_task_adapter
    preflight_model(MODEL)
    ensure_default_task_adapter(require=True)
print("LIVE =", LIVE, "| model =", MODEL,
      "| iterations =", RUN_ITERATIONS, "| candidates =", NUM_CANDIDATES,
      "| examples =", MAX_EXAMPLES, "| hard_examples =", HARD_MAX_EXAMPLES,
      "| seeds =", SEEDS, "| diagnostic_seeds =", DIAGNOSTIC_SEEDS,
      "| eval_calls =", MAX_EVAL_CALLS, "| capability_eval_calls =", CAPABILITY_EVAL_CALLS,
      "| output_root =", OUTPUT_ROOT)


LIVE = True | model = gpt-5.4-nano | iterations = 2 | candidates = 2 | examples = 8 | hard_examples = 4 | seeds = [0, 1, 2, 3, 4] | diagnostic_seeds = [0, 1, 2, 3, 4] | eval_calls = 48 | capability_eval_calls = 96 | output_root = /home/xav/code/Trace/examples/notebook_outputs/recursive_opt_use_cases/three_way_n3_live_20260624_100922


In [3]:
# ======================== EXPERIMENT HARNESS =============================
# One place that runs a spec, collects initial/final scores and artifact refs
# across seeds, and renders comparison tables. Every use case reuses this.
from opto.features.recursive_opt import run_spec, make_level_spec, MemoryLite
from opto.features.recursive_opt.budget import RecursiveOptBudget, reset_budget


def budget_block():
    """Standard budget dict from the config knobs above (spec['budget'] keys)."""
    return {"wall_time_s": WALL_TIME_S, "optimizer_llm_calls": MAX_OPTIMIZER_CALLS,
            "eval_llm_calls": MAX_EVAL_CALLS, "candidates": MAX_CANDIDATES,
            "on_exceed": "return_best"}


def make_budget():
    """Finite budget for direct optimize() calls outside run_spec."""
    return RecursiveOptBudget(
        max_wall_time_s=WALL_TIME_S,
        max_optimizer_llm_calls=MAX_OPTIMIZER_CALLS,
        max_eval_llm_calls=MAX_EVAL_CALLS,
        max_candidates=MAX_CANDIDATES,
        stop_policy="return_best",
    )


def reset_standard_budget():
    """Reset direct optimize() calls to the same envelope as spec runs."""
    reset_budget(make_budget())


def memory_path(name):
    """Return an experiment memory path under the common OUTPUT_ROOT."""
    safe = str(name).strip().strip("./") or "mem"
    return str(OUTPUT_ROOT / safe)


def write_experiment_json(root, filename, payload):
    """Persist the exact experiment spec/config next to reusable artifacts."""
    path = Path(root) / filename
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(payload, indent=2, sort_keys=True, default=str) + "\n")
    return str(path)


def tracebench_block(max_examples=None, inner_steps=None, timeout_seconds=None, eval_kwargs=None):
    """Standard real-adapter bounds (spec['tracebench'] keys).

    Keep this centralized so hard-task experiments can use fewer examples without
    changing the global notebook budget or relying on environment variables.
    """
    block = {"max_examples": int(max_examples or MAX_EXAMPLES),
             "inner_steps": INNER_STEPS if inner_steps is None else int(inner_steps),
             "timeout_seconds": TIMEOUT_S if timeout_seconds is None else int(timeout_seconds)}
    if eval_kwargs:
        block["eval_kwargs"] = dict(eval_kwargs)
    return block


def _one_line_error(exc):
    """Compact, table-safe error summary using the first non-empty message line."""
    lines = [line.strip() for line in str(exc).splitlines() if line.strip()]
    detail = lines[0][:160] if lines else repr(exc)[:160]
    return f"{type(exc).__name__}: {detail}"


def _finite(values):
    """Return finite float values only."""
    out = []
    for value in values:
        try:
            f = float(value)
        except (TypeError, ValueError):
            continue
        if math.isfinite(f):
            out.append(f)
    return out


def _fmt(value):
    """Format optional numeric values for markdown tables."""
    if value is None:
        return "-"
    try:
        f = float(value)
    except (TypeError, ValueError):
        return str(value)
    return f"{f:.3f}" if math.isfinite(f) else "-"


def _md_cell(value: object) -> str:
    """Escape text for a single markdown table cell."""
    text = "-" if value is None else str(value)
    return text.replace("\n", "<br>").replace("|", "\\|")


def _md_code(value: object) -> str:
    """Render a markdown table cell as inline code without breaking pipes."""
    text = _md_cell(value).replace("`", "\\`")
    return f"`{text}`"


def _compact_markdown_tables(markdown: str) -> str:
    """Remove blank lines that would terminate an active markdown table."""
    lines = markdown.splitlines()
    compacted = []
    for index, line in enumerate(lines):
        if line.strip() == "" and compacted and compacted[-1].lstrip().startswith("|"):
            next_index = index + 1
            while next_index < len(lines) and lines[next_index].strip() == "":
                next_index += 1
            if next_index < len(lines) and lines[next_index].lstrip().startswith("|"):
                continue
        compacted.append(line)
    return "\n".join(compacted)


def _display_markdown(markdown: str) -> None:
    """Display markdown after applying table-safety normalization."""
    display(Markdown(_compact_markdown_tables(markdown)))


def _artifact_file(root):
    """Return the JSONL file where reusable artifacts are persisted."""
    return str(Path(root) / "artifacts.jsonl")


def _artifact_ref(root, artifact_id=None):
    """Reference the persisted artifact by file plus optional artifact id."""
    path = _artifact_file(root)
    return f"{path}#{artifact_id}" if artifact_id else path


def _turn_from_artifact_id(artifact_id):
    """Best-effort MemoryLite artifact version from an artifact id."""
    parts = str(artifact_id or "").split(":")
    if len(parts) < 3:
        return None
    try:
        return int(parts[-2])
    except (TypeError, ValueError):
        return None


def _artifact_turn(record):
    """Return the persisted artifact version, not the Trainer step."""
    if not record:
        return None
    try:
        return int(record.get("iteration"))
    except (AttributeError, TypeError, ValueError):
        return _turn_from_artifact_id(record.get("artifact_id") if isinstance(record, dict) else None)


def _fmt_turn(value):
    """Format an optional artifact-version counter."""
    if value is None:
        return "-"
    try:
        return str(int(value))
    except (TypeError, ValueError):
        return str(value)


def _best_step_from_progress(progress, key="best_objective_at"):
    """Return the level step where the configured best score appeared."""
    if not isinstance(progress, dict):
        return None
    point = progress.get(key)
    if not isinstance(point, dict):
        return None
    return point.get("level_step")


def _artifact_version(result_or_row):
    """Return artifact lineage version from explicit metadata or artifact ref."""
    if not isinstance(result_or_row, dict):
        return None
    version = result_or_row.get("artifact_version")
    if version is not None:
        return version
    return _turn_from_artifact_id(result_or_row.get("artifact_id") or result_or_row.get("artifact_file"))


def _result_mean(result):
    """Mean final score for a result dict, or None when no run succeeded."""
    scores = _finite(result.get("scores", []))
    return statistics.mean(scores) if scores else None


def _result_delta(result):
    """Return mean-minus-initial when both values are finite."""
    mean = _result_mean(result)
    initial = result.get("initial") if isinstance(result, dict) else None
    if mean is None or initial is None:
        return None
    try:
        return float(mean) - float(initial)
    except (TypeError, ValueError):
        return None


def _result_eval_calls(result):
    """Best-effort count of real evaluations/trials backing a result row."""
    if not isinstance(result, dict):
        return None
    calls = result.get("eval_calls")
    if calls is not None:
        return calls
    progress = result.get("progress")
    if isinstance(progress, list):
        return len(progress)
    if isinstance(progress, dict) and isinstance(progress.get("history"), list):
        return len(progress["history"])
    return None


def initial_score_for_spec(spec, level_id=None, run_name=None):
    """Evaluate the unoptimized seed artifact once, using the same real adapter bounds."""
    if not LIVE:
        return None, None
    import opto.features.recursive_opt.spec as spec_mod
    try:
        reset_standard_budget()
        lid = level_id or spec["levels"][-1]["id"]
        base_root = Path(run_name or spec.get("memory_root", "mem")).name
        probe_spec = {**spec, "memory_root": memory_path(f"_initial_probes/{base_root}")}
        spec_mod.validate_spec(probe_spec)
        if "tracebench" in probe_spec:
            from opto.features.recursive_opt import tracebench as TB
            TB.configure_tracebench_adapter(probe_spec.get("tracebench") or {}, require=True)
        families = probe_spec.get("families", {})
        memory = MemoryLite(root=probe_spec["memory_root"])
        for level_spec in probe_spec["levels"]:
            level = spec_mod.compile_level(level_spec, memory, families, probe_spec.get("scoring"))
            if level_spec["id"] == lid:
                score, _data = spec_mod._final_eval(level, level_spec, families)
                score = spec_mod._clamp(score, spec_mod._clip_bounds(probe_spec.get("scoring")))
                return float(score), None
        return None, f"level {lid!r} not found"
    except Exception as exc:
        return None, _one_line_error(exc)


def run_spec_seeds(spec, seeds=SEEDS, level_id=None, run_name=None):
    """Run a spec across seeds and keep failures visible without stopping the suite."""
    scores, walls, artifact, aid, errors = [], [], None, None, []
    artifact_ref, best_score, best_step, artifact_version, best_progress = None, None, None, None, None
    lid = level_id or spec["levels"][-1]["id"]
    base_root = Path(run_name or spec.get("memory_root", "mem")).name
    spec_files, best_spec_file = [], None
    if not LIVE:
        for seed in seeds:
            root = memory_path(f"{base_root}_{seed}")
            run_spec_payload = {**spec, "memory_root": root}
            spec_files.append(write_experiment_json(root, "spec.json", run_spec_payload))
        if spec_files:
            best_spec_file = spec_files[0]
        return {"scores": [], "initial": None, "wall_s": None,
                "artifact": "(offline preflight: set LIVE=True to optimize)",
                "artifact_id": None, "artifact_file": None, "best_step": None,
                "artifact_version": None, "progress": None,
                "spec_file": best_spec_file, "dry": True}

    initial, initial_error = initial_score_for_spec(spec, level_id=lid, run_name=base_root)
    if initial_error:
        errors.append(f"initial: {initial_error}")
    for seed in seeds:
        try:
            reset_standard_budget()
            root = memory_path(f"{base_root}_{seed}")
            run_spec_payload = {**spec, "memory_root": root}
            spec_file = write_experiment_json(root, "spec.json", run_spec_payload)
            spec_files.append(spec_file)
            out = run_spec(run_spec_payload)
            r = out["results"][lid]
            score = float(r["score"])
            scores.append(score); walls.append(float(r["wall_s"]))
            ref = _artifact_ref(root, r.get("artifact_id"))
            if best_score is None or score > best_score:
                best_score = score
                artifact, aid, artifact_ref = r["artifact"], r.get("artifact_id"), ref
                best_progress = r.get("progress") or {}
                best_step = _best_step_from_progress(best_progress)
                artifact_version = _turn_from_artifact_id(r.get("artifact_id"))
                best_spec_file = spec_file
        except Exception as exc:
            errors.append(f"seed {seed}: {_one_line_error(exc)}")
    return {"scores": scores, "initial": initial,
            "wall_s": round(statistics.mean(walls), 1) if walls else None,
            "artifact": artifact or "(no successful seed)", "artifact_id": aid,
            "artifact_file": artifact_ref, "best_step": best_step,
            "artifact_version": artifact_version, "progress": best_progress,
            "spec_file": best_spec_file or (spec_files[-1] if spec_files else None),
            "errors": errors, "dry": False}

def _notes_for_result(result: dict[str, object]) -> str:
    """Return compact interpretation, artifact, and error notes for result tables."""
    notes = []
    if result.get("control_reason"):
        notes.append(str(result["control_reason"]))
    if result.get("notes"):
        notes.append(str(result["notes"]))
    artifact = result.get("artifact")
    if artifact and "best_config=" in str(artifact):
        notes.append(str(artifact))
    errors = list(result.get("errors", []) or [])
    notes.extend(str(error) for error in errors[:2])
    if len(errors) > 2:
        notes.append(f"+{len(errors)-2} more")
    return "; ".join(notes)


def summarize(rows):
    """rows: list of (label, result_dict). Returns a markdown comparison table."""
    head = ("| experiment | initial | mean score | delta | std | n | wall_s | eval/trials | best step | artifact version | best artifact file | spec file | notes |\n"
            "|---|---:|---:|---:|---:|---:|---:|---:|---:|---:|---|---|---|")
    lines = [head]
    for label, r in rows:
        if r.get("dry"):
            lines.append(f"| {_md_cell(label)} | - | offline | - | - | 0 | - | - | - | - | - | - | set LIVE=True |")
            continue
        scores = _finite(r.get("scores", []))
        mean = statistics.mean(scores) if scores else None
        std = statistics.pstdev(scores) if len(scores) > 1 else None
        delta = (mean - r["initial"]) if mean is not None and r.get("initial") is not None else None
        notes = _notes_for_result(r)
        artifact_file = r.get("artifact_file") or "-"
        spec_file = r.get("spec_file") or "-"
        lines.append(f"| {_md_cell(label)} | {_fmt(r.get('initial'))} | {_fmt(mean)} | {_fmt(delta)} | "
                     f"{_fmt(std)} | {len(scores)} | {_fmt(r.get('wall_s'))} | {_fmt_turn(_result_eval_calls(r))} | {_fmt_turn(r.get('best_step'))} | {_fmt_turn(_artifact_version(r))} | {_md_code(artifact_file)} | "
                     f"{_md_code(spec_file)} | {_md_cell(notes)} |")
    return "\n".join(lines)


def mark_control(result: dict[str, object], reason: str) -> dict[str, object]:
    """Mark a valid experiment as a diagnostic/control rather than a best-arm candidate."""
    out = dict(result)
    out["exclude_best"] = True
    out["control_reason"] = reason
    return out


def best_of(rows):
    """Return the most informative best row: mean score, then gain, then lower wall time."""
    scored = [(l, r) for l, r in rows if _result_mean(r) is not None]
    informative = [(l, r) for l, r in scored if not r.get("exclude_best")]
    if informative:
        scored = informative
    if not scored:
        return None
    def key(row):
        _label, result = row
        mean = _result_mean(result)
        initial = result.get("initial")
        delta = mean - initial if mean is not None and initial is not None else 0.0
        return (mean, delta, -(result.get("wall_s") or 1e9))
    return max(scored, key=key)


def best_gain_of(rows):
    """Return the best positive non-control gain row, independent of final score."""
    candidates = []
    for label, result in rows:
        if result.get("exclude_best"):
            continue
        delta = _result_delta(result)
        if delta is not None and delta > 0:
            candidates.append((label, result))
    if not candidates:
        return None
    def key(row):
        _label, result = row
        return (_result_delta(result) or 0.0, _result_mean(result) or float("-inf"),
                -(result.get("wall_s") or 1e9))
    return max(candidates, key=key)


from IPython.display import Markdown, display

def show_table(title, rows):
    display(Markdown(f"### {title}\n" + summarize(rows)))
    b = best_of(rows)
    g = best_gain_of(rows)
    if b:
        _display_markdown(f"**Best final score: `{_md_cell(b[0])}`** — best step: `{_fmt_turn(b[1].get('best_step'))}` — artifact version: `{_fmt_turn(_artifact_version(b[1]))}` — artifact file: {_md_code(b[1].get('artifact_file') or '-')} — spec file: {_md_code(b[1].get('spec_file') or '-')}")
        print(textwrap.shorten(str(b[1]["artifact"]), 1200, placeholder=" ...[truncated]"))
    if g and (not b or g[0] != b[0]):
        _display_markdown(f"**Best positive non-control gain: `{_md_cell(g[0])}`** — delta: `{_fmt(_result_delta(g[1]))}` — artifact file: {_md_code(g[1].get('artifact_file') or '-')}")


def _read_jsonl(path):
    """Read JSONL or pretty JSON artifact files as dict records only."""
    p = Path(path)
    if not p.exists():
        return []
    text = p.read_text()
    if p.suffix == ".json":
        try:
            data = json.loads(text)
        except json.JSONDecodeError:
            return []
        if isinstance(data, list):
            return [item for item in data if isinstance(item, dict)]
        return [data] if isinstance(data, dict) else []
    rows = []
    for line in text.splitlines():
        if line.strip():
            try:
                item = json.loads(line)
            except json.JSONDecodeError:
                continue
            if isinstance(item, dict):
                rows.append(item)
    return rows


def _best_artifact_from_dir(mem_dir):
    """Best finite artifact record in a MemoryLite directory."""
    records = _read_jsonl(Path(mem_dir) / "artifacts.jsonl")
    valid = []
    for record in records:
        score = _finite([record.get("score")])
        if score:
            valid.append(record)
    return max(valid, key=lambda r: float(r["score"])) if valid else None


def _best_step_from_artifact(record):
    """Best objective level-step stored in a new artifact's progress metadata."""
    metrics = record.get("metrics") if isinstance(record, dict) else None
    if not isinstance(metrics, dict):
        return None
    return _best_step_from_progress(metrics.get("progress"))


def _initial_from_dir(mem_dir):
    """Best-effort initial score from persisted artifact/episode records."""
    for filename in ("artifacts.jsonl", "episodes.jsonl"):
        records = _read_jsonl(Path(mem_dir) / filename)
        for record in records:
            metrics = record.get("metrics") if isinstance(record, dict) else None
            if isinstance(metrics, dict):
                history = metrics.get("score_history")
                if isinstance(history, list):
                    scores = _finite(history[:1])
                    if scores:
                        return scores[0]
                scores = _finite([metrics.get("initial")])
                if scores:
                    return scores[0]
            scores = _finite([record.get("score")])
            if scores:
                return scores[0]
    return None


UC_DIR_PREFIXES = {
    "UC1 component code": "mem_uc1",
    "UC2 setup/config": "mem_uc2",
    "UC3 capability": "mem_uc3",
    "UC4 family/transfer": "mem_uc4",
    "UC5 optimizer/tool": "mem_uc5",
    "UC6 trace feedback": "mem_uc6",
    "UC7 graph/suboptimizer": ("mem_suboptimizer_graph", "mem_conditional_suboptimizer_graph"),
    "UC8 campaign policy": "mem_uc8",
    "UC9 agentic trace policy": "mem_uc9",
    "UC13 numeric config": "mem_uc13",
}


def _uc_prefixes(prefix):
    """Normalize one or many memory-dir prefixes for historical scans."""
    return tuple(prefix) if isinstance(prefix, (list, tuple)) else (prefix,)


def summarize_past_runs(base_dir=None):
    """Scan previous notebook output folders and summarize persisted artifact scores."""
    base = Path(base_dir or OUTPUT_ROOT.parent)
    rows = []
    for run_dir in sorted([p for p in base.iterdir() if p.is_dir()]):
        for uc_name, prefix in UC_DIR_PREFIXES.items():
            mem_dirs = sorted(
                p
                for item in _uc_prefixes(prefix)
                for p in run_dir.glob(f"{item}*")
                if p.is_dir()
            )
            if not mem_dirs:
                continue
            best, best_dir = None, None
            finals, initials = [], []
            for mem_dir in mem_dirs:
                init = _initial_from_dir(mem_dir)
                if init is not None:
                    initials.append(init)
                art = _best_artifact_from_dir(mem_dir)
                if art is None:
                    continue
                score = float(art["score"])
                finals.append(score)
                if best is None or score > float(best["score"]):
                    best, best_dir = art, mem_dir
            if best is None:
                continue
            rows.append({
                "run": run_dir.name,
                "use_case": uc_name,
                "initial_mean": statistics.mean(initials) if initials else None,
                "best_score": float(best["score"]),
                "final_mean": statistics.mean(finals) if finals else None,
                "n_dirs": len(mem_dirs),
                "best_step": _best_step_from_artifact(best),
                "artifact_version": _artifact_turn(best),
                "artifact_file": _artifact_ref(best_dir, best.get("artifact_id")),
            })
    return rows




def _experiment_from_mem_dir(mem_dir):
    """Compact experiment label inferred from a persisted MemoryLite directory."""
    name = Path(mem_dir).name
    parts = name.split("_")
    if parts and parts[-1].isdigit():
        name = "_".join(parts[:-1])
    return name


def summarize_past_experiments(base_dir=None):
    """Scan every past memory folder, not only the best use-case aggregate."""
    base = Path(base_dir or OUTPUT_ROOT.parent)
    rows = []
    for run_dir in sorted([p for p in base.iterdir() if p.is_dir()]):
        for uc_name, prefix in UC_DIR_PREFIXES.items():
            for mem_dir in sorted(
                p
                for item in _uc_prefixes(prefix)
                for p in run_dir.glob(f"{item}*")
                if p.is_dir()
            ):
                art = _best_artifact_from_dir(mem_dir)
                if art is None:
                    continue
                rows.append({
                    "run": run_dir.name,
                    "use_case": uc_name,
                    "experiment": _experiment_from_mem_dir(mem_dir),
                    "initial": _initial_from_dir(mem_dir),
                    "best_score": float(art["score"]),
                    "best_step": _best_step_from_artifact(art),
                    "artifact_version": _artifact_turn(art),
                    "artifact_file": _artifact_ref(mem_dir, art.get("artifact_id")),
                })
    return rows


def past_experiments_table(rows, limit=None):
    """Render every persisted experiment across all past notebook runs."""
    head = "| run | use case | experiment | initial | best score | best step | artifact version | best artifact file |\n|---|---|---|---:|---:|---:|---:|---|"
    lines = [head]
    ordered = sorted(rows, key=lambda r: (r["run"], r["use_case"], r["experiment"]))
    selected = ordered if limit is None else ordered[-limit:]
    for row in selected:
        lines.append(f"| {_md_cell(row['run'])} | {_md_cell(row['use_case'])} | {_md_cell(row['experiment'])} | "
                     f"{_fmt(row['initial'])} | {_fmt(row['best_score'])} | {_fmt_turn(row.get('best_step'))} | {_fmt_turn(_artifact_version(row))} | "
                     f"{_md_code(row['artifact_file'])} |")
    if limit is not None and len(ordered) > limit:
        lines.append(f"| ... | ... | {len(ordered)-limit} older rows omitted | - | - | - | - | - |")
    return "\n".join(lines)

def past_runs_table(rows):
    """Render the cross-run artifact summary."""
    head = "| run | use case | initial mean | final mean | best score | best step | artifact version | n memory dirs | best artifact file |\n|---|---|---:|---:|---:|---:|---:|---:|---|"
    lines = [head]
    for row in rows:
        lines.append(f"| {_md_cell(row['run'])} | {_md_cell(row['use_case'])} | {_fmt(row['initial_mean'])} | "
                     f"{_fmt(row['final_mean'])} | {_fmt(row['best_score'])} | {_fmt_turn(row.get('best_step'))} | {_fmt_turn(_artifact_version(row))} | {row['n_dirs']} | "
                     f"{_md_code(row['artifact_file'])} |")
    return "\n".join(lines)

print("harness ready")


harness ready


---
### Root-cause diagnostics
These quick checks separate optimizer behavior from benchmark shape. They are intentionally small: probe score spread tells us whether a surface has enough room to learn, while code-baseline probes show when UC1/UC5 are saturated by a narrow deterministic validator rather than by broad benchmark performance. BBEH is probed here for task-shape awareness, but it is not used in UC3 because its Trace-Bench bundle is raw/code-artifact style rather than a prompt-capability surface. UC2 now includes a fixed mixed task set so easy and harder prompt examples can be scored together instead of relying on a single saturated task.


In [ ]:
from opto.features.recursive_opt.spec import score_spread
from opto.features.recursive_opt.tracebench import make_code_evaluator, configure_tracebench_adapter

if LIVE:
    configure_tracebench_adapter(tracebench_block(), require=True)
    diagnostic_rows = []
    probe_prompts = [
        {},
        {"starting_artifact": "Answer directly."},
        {"starting_artifact": "Plan step by step, then verify the answer before replying."},
    ]
    for task in ["internal:multiobjective_gsm8k", "internal:multiobjective_bbeh", "hf:drop", "hf:qasper"]:
        try:
            spread = score_spread(task, probes=probe_prompts)
            scores = [r.get("score") for r in spread["rows"]]
            diagnostic_rows.append((task, spread["valid_spread"], spread["invalid_probes"], scores))
        except Exception as exc:
            diagnostic_rows.append((task, None, None, _one_line_error(exc)))

    code_probe = make_code_evaluator("internal:batch_design", "batch_design")
    code_rows = []
    for label, fn in [
        ("take_first", lambda n, k: list(range(k))),
        ("take_last", lambda n, k: list(range(n-k, n))),
        ("stride", lambda n, k: list(range(0, n, max(1, n//k)))[:k]),
        ("hard_mod3", lambda n, k: [i for i in range(n) if i % 3 == 0][:k]),
    ]:
        score, feedback = code_probe(lambda **kw: fn(**kw), "internal:batch_design")
        code_rows.append((label, score, feedback))

    lines = ["| probe | spread/score | details |", "|---|---:|---|"]
    for task, spread, invalid, scores in diagnostic_rows:
        lines.append(f"| {task} score spread | {_fmt(spread)} | invalid={invalid}; scores={scores} |")
    for label, score, feedback in code_rows:
        lines.append(f"| batch_design baseline `{label}` | {_fmt(score)} | {feedback[:180]} |")
    display(Markdown("\n".join(lines)))
else:
    display(Markdown("Diagnostics skipped: set `LIVE=True`."))


---
## Use Case 1 — Optimize & validate a NEW Trace component (code surface) ⭐ strongest today

**Why:** the code surface has a deterministic evaluator, so the signal is clean and the
before/after is a real diff. Best for: a new trainer hot-path, batch sampler, trace
summarizer, validator, or compact task-solving component.

**Experiments:**
1. **batch_design** on `internal:batch_design` — known-climbable failure-balanced selector.
2. **trace_summarizer** on `internal:code_param` — multi-criteria evaluator (keep error evidence + be concise), with default and stricter prompt variants.
3. **BBEH direct code solver** on real Trace-Bench examples — harder than the toy selectors and saved as reusable Python code.

**Mode:** offline pre-flight proves the surface; **set LIVE=True for the real rewrite.**


In [4]:
# Use Case 1 — code surface. Uses ComponentSpec + CodeArtifactLevel via optimize().
# Interpretation: this proves optimizer-to-code rewriting, validation, rollback, and
# artifact persistence. It is intentionally narrow; saturation means the validator is
# easy, not that the learned component generalizes to all training loops.
from opto.features.recursive_opt import CodeArtifactLevel, ComponentSpec, optimize, RecursiveGuide
from opto.features.recursive_opt.tracebench import make_code_evaluator, make_dataset, make_tracebench_direct_answer_evaluator, make_artifact_emitter_evaluator
import opto.trace as trace


def run_code_experiment(name, task_id, objective, seeds=SEEDS, memory_name=None, baseline=None, evaluate=None, iterations=None, num_candidates=None):
    """One code-surface experiment across isolated memory roots per seed."""
    scores, initial_scores, walls, final_code, errors = [], [], [], None, []
    best_ref, best_score, best_code, best_spec_file, artifact_version = None, None, None, None, None
    for seed in seeds:
        root_name = memory_name or f"mem_uc1_{name}"
        root = memory_path(f"{root_name}_{seed}")
        baseline_fn = baseline or _BASELINES[name]
        local_iterations = int(iterations or RUN_ITERATIONS)
        local_candidates = int(num_candidates or NUM_CANDIDATES)
        payload = {
            "surface": "code",
            "component": name,
            "task_id": task_id,
            "objective": objective,
            "baseline": getattr(baseline_fn, "__name__", str(baseline_fn)),
            "iterations": local_iterations,
            "num_candidates": local_candidates,
            "max_examples": MAX_EXAMPLES,
        }
        spec_file = write_experiment_json(root, "component_spec.json", payload)
        best_spec_file = spec_file
        if not LIVE:
            continue
        try:
            mem = MemoryLite(root=root)
            spec = ComponentSpec(name=name, baseline=baseline_fn,
                                 evaluate=evaluate or make_code_evaluator(task_id, name), objective=objective)
            level = CodeArtifactLevel(spec, memory=mem)
            guide = RecursiveGuide()
            initial_scores.append(float(guide(task_id, level.forward(task_id), None)[0]))
            reset_standard_budget()
            t0 = time.time()
            optimize(level, make_dataset([task_id], repeats=MAX_EXAMPLES), guide=guide,
                     iterations=local_iterations, num_candidates=local_candidates)
            # Code surfaces persist every validated implementation. Report and
            # re-score the best saved artifact so the table points at the reusable
            # solution, even if the Trainer's active slot moved on.
            best = mem.best_artifact(str(task_id), "code")
            if best is not None and level.parameters():
                level.parameters()[0]._data = best.content
            walls.append(round(time.time() - t0, 1))
            score = float(guide(task_id, level.forward(task_id), None)[0])
            if best is not None and float(best.score) >= score:
                score, final_code = float(best.score), best.content
                ref = _artifact_ref(root, best.artifact_id)
                turn = int(best.iteration)
            else:
                final_code = level.current_code()
                ref = _artifact_file(root)
                turn = _turn_from_artifact_id(ref)
            scores.append(score)
            if best_score is None or score > best_score:
                best_score, best_ref, best_code, best_spec_file = score, ref, final_code, spec_file
                artifact_version = turn
        except Exception as exc:
            errors.append(f"seed {seed}: {_one_line_error(exc)}")
    if not LIVE:
        return {"scores": [], "initial": None, "wall_s": None,
                "artifact": "(offline preflight) set LIVE=True to optimize",
                "artifact_id": None, "artifact_file": None, "best_step": None,
                "artifact_version": None, "progress": None,
                "spec_file": best_spec_file, "dry": True,
                "errors": errors}
    return {"scores": scores, "initial": statistics.mean(initial_scores) if initial_scores else None,
            "wall_s": round(statistics.mean(walls), 1) if walls else None,
            "artifact": best_code or final_code or "(no successful seed)", "artifact_id": None,
            "artifact_file": best_ref, "best_step": None,
            "artifact_version": artifact_version, "progress": None,
            "spec_file": best_spec_file, "errors": errors, "dry": False}


def _weak_batch(self, n, k): return list(range(k))
def _trunc_summary(self, trace_text): return str(trace_text)[-500:]
def _bbeh_direct_solver(self, question):
    """Return True/False for a BBEH boolean expression ending with ' is'."""
    return "True"

def _norm_bool_answer(value):
    """Normalize boolean answers for BBEH direct-solver validation."""
    return str(value).strip().lower().replace(".", "").replace(" ", "")

_BASELINES = {"batch_design": _weak_batch, "trace_summarizer": _trunc_summary,
              "bbeh_direct_solver": _bbeh_direct_solver}



## Use Case 2 — Learn the best SETUP / default prompt for a family (config surface)

**Why:** optimize *which existing components & artifact* to use. Under `INNER_STEPS=0`
only **causally-active** fields move score: `starting_artifact`, `initial_knowledge`,
`trace_type`. (Trainer/batch only activate at `INNER_STEPS>0` — the contract enforces this.)

**Experiments:**
1. **GSM8K artifact menu** — search a small set of prompt strategies (incl. empty control arm).
2. **+ initial_knowledge / warm prior** — test whether extra setup context or saved priors improve the same prompt surface.
3. **harder QA controls** — compare DROP (often saturated) with QASPER (less saturated but noisier/slower).
4. **mixed GSM8K+QASPER task set** — test whether learning on easy + harder examples together avoids a prompt that only fits the easy task.

**Mode:** needs LIVE + Trace-Bench (real task scores).


In [5]:
# Use Case 2 - config surface with one causal numeric arm.
# The prompt-only rows remain diagnostics. The causal numeric row turns on
# inner training and targets batch_design/batch_size, so the adapter must consume
# the proposed fields rather than merely echoing a config artifact.

# F3: TRUE numeric-optimizer arm - Optuna over the causal fields via the real inner runner.
# Produces a concrete best-config artifact and the per-trial learning curve (high-value output).
# Returns the same dict shape run_spec_seeds produces so the summary table renders it.
def numeric_search_space(fields, constraints):
    """Return a numeric optimizer search space aligned with spec constraints."""
    return {field: ("cat", tuple(constraints[field]))
            for field in fields if field in constraints}


def numeric_optimizer_arm(task, fields, constraints, *, tasks=None,
                          inner_steps=2, max_examples=6,
                          family_name="numeric_arm", trials=16,
                          memory_root="./mem_numeric_arm"):
    base = {"scores": [], "initial": None, "wall_s": None, "artifact": None,
            "artifact_id": None, "artifact_file": None, "best_step": None,
            "artifact_version": None, "progress": None, "spec_file": None,
            "eval_calls": None, "errors": [], "dry": False}
    if not LIVE:
        return {**base, "dry": True,
                "artifact": "(offline preflight: set LIVE=True to run the numeric optimizer)"}
    try:
        from opto.features.recursive_opt import optimize_config_numeric, MemoryLite
        from opto.features.recursive_opt import spec as _spec
        task_ids = list(tasks or [task])
        spec = config_spec(fields, numeric_constraints=constraints,
                           task=task_ids[0], tasks=task_ids if len(task_ids) > 1 else None,
                           family_name=family_name, max_examples=max_examples,
                           inner_steps=inner_steps, memory_root=memory_root)
        fams = {family_name: task_ids}
        mem = MemoryLite(root=memory_path(memory_root + "_lvl"))
        level = _spec.compile_level(spec["levels"][0], mem, fams)
        eval_label = task_ids[0] if len(task_ids) == 1 else f"task_set:{family_name}"
        t0 = time.time()
        best, score, history = optimize_config_numeric(level, eval_label, fields,
                                                        optimizer="optuna", max_trials=trials,
                                                        space=numeric_search_space(fields, constraints))
        wall_s = round(time.time() - t0, 1)
        eval_calls = len(history)
        curve = [round(float(s), 3) for _, s in history]
        artifact_text = f"best_config={best} | curve={curve}"
        root = memory_path(memory_root + "_lvl")
        artifact_payload = {
            "best_config": best,
            "score": float(score),
            "curve": curve,
            "history": [{"assignment": assignment, "score": float(s)}
                        for assignment, s in history],
            "fields": list(fields),
            "tasks": task_ids,
            "optimizer": "optuna",
            "trials": int(trials),
            "eval_calls": eval_calls,
            "wall_s": wall_s,
        }
        artifact_file = write_experiment_json(root, "numeric_optimizer_result.json", artifact_payload)
        spec_file = write_experiment_json(root, "spec.json", spec)
        return {**base, "scores": [float(score)],
                "initial": curve[0] if curve else None,
                "artifact": artifact_text,
                "artifact_file": artifact_file,
                "wall_s": wall_s,
                "eval_calls": eval_calls,
                "best_step": (max(range(len(curve)), key=lambda k: curve[k]) if curve else None),
                "progress": curve,
                "spec_file": spec_file,
                "notes": f"numeric trials={eval_calls}; zero LLM proposal calls; each trial still runs the real inner evaluator"}
    except Exception as exc:
        return {**base, "errors": [_one_line_error(exc)]}

FAMILY_TASK = "internal:multiobjective_gsm8k"
HARD_PROMPT_TASKS = {"drop": "hf:drop", "qasper": "hf:qasper"}
ART_MENU = ["", "Answer directly.", "Plan step by step, then answer.",
            "Plan step by step, then verify the answer before replying.",
            "Use the provided context as evidence, reason briefly, then answer exactly."]
CAUSAL_NUMERIC_TARGETS = ["batch_design", "batch_size"]
CAUSAL_NUMERIC_CONSTRAINTS = {
    "batch_design": ["random", "failure_balanced", "curriculum", "diversity"],
    "batch_size": [2, 4, 8],  # F1: give the numeric arm a real integer dimension to search
}


def config_spec(targets, reuse=False, extra_constraints=None, numeric_constraints=None,
                memory_root="./mem_uc2", task=FAMILY_TASK, tasks=None,
                family_name="reasoning", max_examples=None, inner_steps=None,
                fixed_overrides=None, budget=None):
    """Build an O1 config spec for one task or a fixed mixed task set."""
    task_ids = list(tasks or [task])
    cons = {"starting_artifact": ART_MENU}
    if extra_constraints:
        cons.update(extra_constraints)
    if numeric_constraints:
        cons.update(numeric_constraints)
    fixed = {"optimizer": "OptoPrimeV2", "trace_type": "internal",
             "credit_horizon": "step", "trainer": "PrioritySearch"}
    if fixed_overrides:
        fixed.update(fixed_overrides)
    level_kwargs = {"task": task_ids[0]} if len(task_ids) == 1 else {"tasks": task_ids}
    return {"families": {family_name: task_ids},
            "memory_root": memory_root, "reuse_priors": reuse,
            "budget": dict(budget or budget_block()),
            "tracebench": tracebench_block(max_examples=max_examples, inner_steps=inner_steps),
            "scoring": {"clip": [-1.0, 1.0]},
            "levels": [make_level_spec(
                id="o1_setup", surface="config", family=family_name, **level_kwargs,
                targets=targets, constraints=cons, fixed=fixed,
                iterations=RUN_ITERATIONS)]}



---
## Use Case 3 — Discover a new CAPABILITY from a spec + objectives (capability surface)

**Why:** synthesize a capability artifact (a skill-like text) that satisfies a
natural-language spec while trading off objectives (accuracy↑, cost↓).

**3 experiments:** three different **seed specifications** for the same objective set, to
see which framing the optimizer can push furthest (a complementary-results search).

**Mode:** needs LIVE + Trace-Bench. Uses the `capability` surface in a spec.

In [ ]:
# Use Case 3 — capability surface. Three seed framings; pareto over (accuracy, cost).
# Keep this on prompt-compatible GSM8K. BBEH is intentionally excluded here: the
# diagnostic probe showed it is a raw/code-artifact task for this adapter, so a
# natural-language capability prompt is evaluated as invalid code. That belongs to
# code-artifact experiments, not this prompt-capability surface.
from opto.features.recursive_opt.tracebench import make_multiobjective_evaluator

CAP_TASKS = ["internal:multiobjective_gsm8k"]
CAP_OBJECTIVES = {"accuracy": "max", "cost": "min"}
_cap_evaluator = make_multiobjective_evaluator(
    CAP_TASKS,
    CAP_OBJECTIVES,
    required_terms=("plan", "verify"),
)


def capability_spec(seed_text, memory_root="./mem_uc3"):
    cap_budget = {**budget_block(), "eval_llm_calls": CAPABILITY_EVAL_CALLS}
    return {"families": {"reasoning": CAP_TASKS}, "memory_root": memory_root,
            "budget": cap_budget, "tracebench": tracebench_block(),
            "levels": [ make_level_spec(
                id="cap", surface="capability", family="reasoning", task=CAP_TASKS[0],
                seed=seed_text, evaluator=_cap_evaluator,
                objective_config={"mode": "pareto", "minimize": ["cost"]},
                iterations=RUN_ITERATIONS)]}

uc3 = [
  ("seed: weak (constant-answer headroom)", run_spec_seeds(capability_spec(
       "Always answer 0. Do not plan, verify, decompose, or explain.", "./mem_uc3_weak"), seeds=SEEDS,
       level_id="cap", run_name="mem_uc3_weak")),  # F2: deliberately low baseline => real headroom
  ("seed: terse",   run_spec_seeds(capability_spec("Solve correctly using the fewest words.", "./mem_uc3_terse"),
                                   run_name="mem_uc3_terse")),
  ("seed: verify",  run_spec_seeds(capability_spec("Make a short plan; solve; then verify/check the answer before replying.", "./mem_uc3_verify"),
                                   run_name="mem_uc3_verify")),
  ("seed: decompose", run_spec_seeds(capability_spec("Plan, decompose into sub-steps, solve each, then verify before answering.", "./mem_uc3_decompose"),
                                     run_name="mem_uc3_decompose")),
]
show_table("Use Case 3 — capability discovery", uc3)


---
## Use Case 4 — Family policy (O2) & transferable prior (O3) — EXPERIMENTAL

**Why:** learn config per family (O2) and induce a prior validated on held-out families (O3).
This is mechanically real but still noisy: treat results as exploratory and require warm>cold
by more than run noise before believing transfer.

**Experiments:**
1. **O2 only** — one family-policy level over a small mixed task set.
2. **O2→O3 cold** — add prior induction with no prior reuse.
3. **O2→O3 warm** — re-run with prior reuse to measure transfer.

**Mode:** needs LIVE. The mixed task set intentionally includes non-saturated QASPER so transfer is not judged only on saturated controls.


In [6]:
# Use Case 4 - O2/O3 transfer diagnostic plus one causal numeric policy arm.
# Warm-prior rows test transfer. The numeric O2 row turns on inner_steps=2 so the
# family-policy surface has at least one adapter-consumed field to optimize.
def family_policy_spec(kind="o2", warm=False, targets=None, constraints=None,
                       inner_steps=None, memory_root="./mem_uc4"):
    """Build an O2/O3 family-policy spec with optional active numeric fields."""
    fams = {"gsm8k": [FAMILY_TASK], "qasper": [HARD_PROMPT_TASKS["qasper"]]}
    target_fields = list(targets or ["starting_artifact"])
    cons = dict(constraints or {"starting_artifact": ART_MENU})
    levels = [make_level_spec(
        id="o2_policy", surface="family_policy", family="*", families=list(fams),
        targets=target_fields, constraints=cons,
        fixed={"optimizer": "OptoPrimeV2", "trainer": "PrioritySearch",
               "trace_type": "internal", "credit_horizon": "step"},
        iterations=RUN_ITERATIONS)]
    if kind == "o3":
        levels.append(make_level_spec(
            id="o3_prior", surface="prior", family="*", task=HARD_PROMPT_TASKS["qasper"],
            targets=target_fields, constraints=cons,
            fixed={"optimizer": "OptoPrimeV2", "trainer": "PrioritySearch",
                   "trace_type": "internal", "credit_horizon": "step"},
            iterations=RUN_ITERATIONS))
    return {"families": fams, "memory_root": memory_root, "reuse_priors": warm,
            "budget": budget_block(),
            "tracebench": tracebench_block(max_examples=HARD_MAX_EXAMPLES, inner_steps=inner_steps),
            "scoring": {"clip": [-1.0, 1.0]}, "levels": levels}



## Use Case 5 — Code helpers vs optimizer-side tools — EXPERIMENTAL

**Why:** there are three distinct meanings of “tool” here, and the notebook measures them separately.

**Code-helper optimization:** model a helper/selector as `CodeArtifactLevel`; the LLM rewrites
the component code and the reusable solution is saved as `kind="code"` in `artifacts.jsonl`.

**Optimizer-side tool calling:** `AgenticOptimizer` calls registered helper tools such as
`note` or `trace_search` before proposing an update, then injects their evidence into optimizer
feedback. This changes the optimizer's context; it does not give downstream agent tools to the
optimized artifact.

**Tool-policy artifact:** a separate code-surface arm learns a compact policy that selects which
optimizer tools are useful from the task signal. That artifact can be reused as an input policy for
optimizer-side tool calling.

**Mode:** offline pre-flight + LIVE for real rewrites/tool-feedback proposals. Saturated helper-code controls remain visible but are not selected as the most informative best arm.


In [7]:

# Use Case 5 — code helpers vs optimizer-side tools.
# v4 clarified the split: reusable code/tool-policy artifacts are high signal;
# fixed optimizer-side tool-call configs are slow diagnostics and mostly save config.
from opto.features.recursive_opt import parse_optimizer_tool_policy

def _baseline_take_last(self, n, k):  return list(range(n-k, n))
def _baseline_stride(self, n, k):     return list(range(0, n, max(1, n//k)))[:k]
def _baseline_tool_policy(self, signal): return "tools: note"

OPTIMIZER_TOOL_NAMES = ("trace_search", "run_subset", "artifact_linter", "note")
TOOL_POLICY_CASES = [
    {"signal": "Need prior failures and family examples before proposing a prompt update.",
     "required": {"trace_search", "note"}},
    {"signal": "Need validate a candidate on a small subset before accepting it.",
     "required": {"run_subset"}},
    {"signal": "Need inspect the saved artifact for syntax and current_code reuse.",
     "required": {"artifact_linter"}},
    {"signal": "Saturated control: record a note and do not spend expensive tool calls.",
     "required": {"note"}, "forbidden": {"trace_search", "run_subset", "artifact_linter"}},
]


def evaluate_optimizer_tool_policy(component, _task_id):
    """Score a generated policy that selects optimizer-side helper tools."""
    scores, feedbacks, selections = [], [], []
    for case in TOOL_POLICY_CASES:
        raw = component(case["signal"])
        selected = parse_optimizer_tool_policy(raw, OPTIMIZER_TOOL_NAMES, max_tools=3)
        selected_set = set(selected)
        required = set(case["required"])
        forbidden = set(case.get("forbidden", set()))
        missing = sorted(required.difference(selected_set))
        extra = sorted(selected_set.difference(required).difference({"note"}))
        forbidden_hit = sorted(selected_set & forbidden)
        coverage = len(required & selected_set) / max(1, len(required))
        score = max(0.0, coverage - 0.15 * len(extra) - 0.35 * len(forbidden_hit))
        scores.append(score)
        selections.append({"signal": case["signal"], "selected": selected, "required": sorted(required)})
        feedbacks.append(
            f"signal={case['signal']!r}; selected={selected}; required={sorted(required)}; "
            f"missing={missing}; extra={extra}; forbidden_hit={forbidden_hit}; score={score:.2f}"
        )
    mean = statistics.mean(scores)
    feedback = " | ".join(feedbacks) + f" | selections={selections}"
    return mean, feedback



_BASELINES["optimizer_tool_policy"] = _baseline_tool_policy


---
## Use Case 6 — Which FEEDBACK CHANNEL helps the optimizer? (trace_type with fixed credit_horizon) — EXPERIMENTAL

**Why:** previous grids mixed too many knobs and saturated on easier tasks. This version fixes
`credit_horizon=step` from earlier evidence, then asks one controlled question: whether
`trace_type` (`internal` / `otel` / `hybrid`) changes optimizer proposals on a non-saturated
real Trace-Bench task.

The task is QASPER by default because the sampled DROP configuration saturated at 1.0 and
therefore could not distinguish trace designs. Scores are real Trace-Bench prompt/config
scores, but small-sample noise remains high.

**Mode:** needs LIVE + Trace-Bench.


In [ ]:
# Use Case 6 - feedback-channel diagnostic plus one causal numeric arm.
# Internal/otel/hybrid compare trace representations with credit_horizon fixed at
# the prior best setting (step). The numeric row controls for whether the config
# surface can improve when adapter-consumed fields are targeted.
UC6_TASK = HARD_PROMPT_TASKS["qasper"]

def feedback_spec(level_id, trace_type, targets=None, constraints=None,
                  inner_steps=None, memory_root=None):
    """Build a feedback-channel config spec for one trace representation."""
    target_fields = list(targets or ["starting_artifact"])
    cons = dict(constraints or {"starting_artifact": ART_MENU})
    root = memory_root or f"./mem_uc6_{level_id}"
    return {"families": {"reasoning": [UC6_TASK]}, "memory_root": root,
            "budget": budget_block(),
            "tracebench": tracebench_block(max_examples=HARD_MAX_EXAMPLES, inner_steps=inner_steps),
            "scoring": {"clip": [-1.0, 1.0]},
            "levels": [make_level_spec(
                id=level_id, surface="config", family="reasoning", task=UC6_TASK,
                targets=target_fields, constraints=cons,
                fixed={"optimizer": "OptoPrimeV2", "trainer": "PrioritySearch",
                       "trace_type": trace_type, "credit_horizon": "step"},
                iterations=RUN_ITERATIONS)]}

uc6 = [(f"trace_type={tt} | credit_horizon=step",
        run_spec_seeds(feedback_spec(f"o1_trace_{tt}", tt), seeds=DIAGNOSTIC_SEEDS,
                       level_id=f"o1_trace_{tt}", run_name=f"mem_uc6_trace_{tt}"))
       for tt in ["internal", "otel", "hybrid"]]
uc6.append(("trace_type=internal | causal numeric config (inner_steps=2)",
            run_spec_seeds(feedback_spec("o1_trace_internal_numeric", "internal",
                                         targets=CAUSAL_NUMERIC_TARGETS,
                                         constraints=CAUSAL_NUMERIC_CONSTRAINTS,
                                         inner_steps=2,
                                         memory_root="./mem_uc6_trace_internal_numeric"),
                           seeds=DIAGNOSTIC_SEEDS,
                           level_id="o1_trace_internal_numeric",
                           run_name="mem_uc6_trace_internal_numeric")))

uc6.append(("trace_type=internal numeric-optimizer (Optuna, inner_steps=2)",
    numeric_optimizer_arm(UC6_TASK, CAUSAL_NUMERIC_TARGETS,
        CAUSAL_NUMERIC_CONSTRAINTS, family_name="uc6_internal_numeric",
        memory_root="./mem_uc6_internal_numeric")))  # F3

show_table("Use Case 6 - trace representation diagnostic", uc6)


---
## Master summary — all use cases at a glance

Run after the experiments above. The first table shows **every current-run experiment** with
initial score, mean score, delta, wall time, best optimizer step, artifact version, and the file/id of the best saved artifact. The
second table picks one best non-control row per use case; interpret saturated rows with the guardrails below.
The historical tables scan all persisted `examples/notebook_outputs/recursive_opt_use_cases`
runs so previous artifacts can be compared and reused. `n memory dirs` counts persisted memory folders
for that use case in that run, usually one folder per experiment arm and seed. `best step` is the
recursive-opt `level_step` from `summary.json` / `metrics['progress']` when available; older artifacts show `-`.
`artifact version` is the MemoryLite lineage counter and remains available for historical folders.


## Use Case 7 — Graph routing to a sub-optimizer tool — PROBE

**Why:** this isolates the “use another optimizer as a tool/sub-optimizer” question from Trace-Bench noise. The first graph starts with a weak draft route and has a deterministic SciPy sub-optimizer node available, proving that the recursive optimizer can learn to call a sub-optimizer. The second graph adds a tool-use cost and mixed easy/hard inputs, so unconditional SciPy use is no longer optimal and the useful target is conditional routing.

**Mode:** needs LIVE because the graph route is selected by the LLM optimizer. The output artifact stores the learned graph parameter, score history, and the spec needed to reproduce the graph probe.


In [ ]:
# Use Case 7 — graph routing to a downstream sub-optimizer tool.
# The first arm proves the optimizer can route to SciPy when the tool is always
# useful. The second arm adds a per-tool cost and mixed easy/hard cases, so the
# useful behavior is conditional routing rather than unconditional tool use.
from argparse import Namespace
try:
    from examples.recursive_opt_abc_probe import (  # type: ignore
        run_suboptimizer_graph,
        run_conditional_suboptimizer_graph,
    )
    _UC7_ENABLED = True
    _UC7_IMPORT_ERROR = None
except Exception as exc:  # pragma: no cover - runtime dependency guard
    run_suboptimizer_graph = None
    run_conditional_suboptimizer_graph = None
    _UC7_ENABLED = False
    _UC7_IMPORT_ERROR = str(exc)


def run_suboptimizer_use_case(runner, artifact_id, reason):
    """Run a graph/suboptimizer probe and return a table-compatible result."""
    if not LIVE:
        return {
            "scores": [], "initial": None, "wall_s": None,
            "artifact": "(offline preflight: set LIVE=True to optimize graph route)",
            "artifact_id": None, "artifact_file": None, "spec_file": None, "dry": True,
        }
    if not _UC7_ENABLED or runner is None:
        return {
            "scores": [], "initial": None, "wall_s": None,
            "artifact": "(skipped: missing langgraph/probe dependencies)",
            "artifact_id": None, "artifact_file": None, "spec_file": None,
            "errors": [f"UC7 unavailable: {_UC7_IMPORT_ERROR}"], "dry": False,
        }
    reset_standard_budget()
    args = Namespace(model=MODEL, iterations=RUN_ITERATIONS, candidates=NUM_CANDIDATES,
                     max_examples=MAX_EXAMPLES, timeout_seconds=TIMEOUT_S,
                     live=True, skip_preflight=True)
    result = runner(OUTPUT_ROOT, args)
    artifact = json.dumps({
        "params": result.get("params"),
        "score_history": result.get("score_history"),
        "oracle_tool_score": result.get("oracle_tool_score"),
        "always_tool_score": result.get("always_tool_score"),
    }, indent=2, sort_keys=True)
    return {
        "scores": [float(result["final"])],
        "initial": float(result["initial"]),
        "wall_s": float(result["wall_s"]),
        "artifact": artifact,
        "artifact_id": artifact_id,
        "artifact_file": result.get("artifact_file"),
        "spec_file": result.get("spec_file"),
        "errors": [],
        "control_reason": reason,
    }

uc7 = [
    ("graph route: always-use SciPy suboptimizer", run_suboptimizer_use_case(
        run_suboptimizer_graph, "graph:suboptimizer:latest", "learned graph route to SciPy sub-optimizer")),
    ("graph route: conditional cost-aware suboptimizer", run_suboptimizer_use_case(
        run_conditional_suboptimizer_graph, "graph:conditional_suboptimizer:latest", "tests conditional routing under tool cost")),
]
show_table("Use Case 7 — graph/suboptimizer routing", uc7)


---
## Use Case 8 — Meta-campaign policy: dataset mix, saturation, stall/restart

**Why:** the latest runs showed that the most important meta decision is often *not* another optimizer step. The controller should decide when a task is saturated, when a harder task has enough signal, when a mixed dataset is harmful, and when to restart/switch rather than keep spending LLM calls.

This use case optimizes an executable campaign policy. It is a reverse experiment for the least useful arms: saturated DROP/stride and low-spread GSM8K are turned into decision cases where the correct behavior is to stop, mark as control, or switch dataset.

**Mode:** LIVE code-surface rewrite. It is intentionally fast and structured; the output is reusable policy code saved in `artifacts.jsonl`.


In [8]:

# Use Case 8 — meta-campaign/dataset policy.
# v4 proved the surface works but the evaluator was too permissive: weak policies
# that said "continue" too often still scored ~0.4. This stricter evaluator rewards
# the exact control action, task choice, budget, and reason so useful policies are
# materially different from the seed.
def _baseline_campaign_policy(self, diagnostics):
    """Return action/task/reason from diagnostics; this seed is intentionally weak."""
    return "action: continue\ntask: internal:multiobjective_gsm8k\nmax_examples: 8\nreason: default"

CAMPAIGN_TASKS = (
    "internal:multiobjective_gsm8k",
    "internal:multiobjective_bbeh",
    "hf:drop",
    "hf:qasper",
    "mixed:gsm8k+qasper",
)

CAMPAIGN_POLICY_CASES = [
    {
        "name": "saturated_drop_control",
        "diagnostics": {"task": "hf:drop", "mean_score": 1.0, "spread": 0.0,
                         "recent_delta": 0.0, "wall_s": 38.0, "saturated": True},
        "actions": {"stop", "skip", "control", "drop"},
        "tasks": set(), "avoid": {"hf:drop"}, "max_examples": (0, 2),
        "reasons": {"satur", "ceiling", "control", "stop"},
    },
    {
        "name": "high_headroom_bbeh_exploit",
        "diagnostics": {"task": "internal:multiobjective_bbeh", "mean_score": 0.625,
                         "spread": 1.0, "recent_delta": 0.375, "wall_s": 4.8, "saturated": False},
        "actions": {"exploit", "train", "continue", "increase"},
        "tasks": {"internal:multiobjective_bbeh"}, "avoid": set(), "max_examples": (8, 16),
        "reasons": {"headroom", "spread", "bbeh", "fast"},
    },
    {
        "name": "qasper_harder_probe",
        "diagnostics": {"task": "hf:qasper", "mean_score": 0.125, "spread": 0.082,
                         "recent_delta": 0.037, "wall_s": 39.2, "saturated": False},
        "actions": {"probe", "explore", "sample", "budget"},
        "tasks": {"hf:qasper"}, "avoid": set(), "max_examples": (3, 6),
        "reasons": {"hard", "qasper", "noisy", "probe"},
    },
    {
        "name": "gsm8k_low_spread_stall",
        "diagnostics": {"task": "internal:multiobjective_gsm8k", "mean_score": -0.148,
                         "spread": 0.042, "recent_delta": 0.002, "wall_s": 71.0, "saturated": False},
        "actions": {"restart", "switch", "probe", "reduce"},
        "tasks": {"hf:qasper", "internal:multiobjective_bbeh"},
        "avoid": {"internal:multiobjective_gsm8k"}, "max_examples": (3, 8),
        "reasons": {"low", "spread", "stall", "switch"},
    },
    {
        "name": "mixed_regressed_split",
        "diagnostics": {"task": "mixed:gsm8k+qasper", "mean_score": -0.010,
                         "spread": 0.008, "recent_delta": -0.006, "wall_s": 69.8,
                         "mixed_regressed": True},
        "actions": {"split", "separate", "restart", "ablate"},
        "tasks": {"hf:qasper", "internal:multiobjective_bbeh"},
        "avoid": {"mixed:gsm8k+qasper"}, "max_examples": (3, 8),
        "reasons": {"mixed", "regress", "separate", "ablate"},
    },
]


def _policy_text(raw):
    """Normalize a generated campaign/tool policy to lowercase text."""
    if isinstance(raw, dict):
        return json.dumps(raw, sort_keys=True).lower()
    return str(raw).lower()


def _mentioned_tasks(text, known_tasks):
    """Return known task ids mentioned in generated policy text."""
    return {task for task in known_tasks if task.lower() in text}


def _keyword_present(text, word):
    """Match policy keywords without treating do_not_promote as promote."""
    import re
    key = str(word).strip().lower()
    if not key:
        return False
    # Short stems (satur/regress) and explicit phrases are intentionally partial.
    if len(key) <= 5 or any(ch in key for ch in " _:/-"):
        return key in text
    return re.search(rf"(?<![a-z0-9_]){re.escape(key)}(?![a-z0-9_])", text) is not None


def _contains_any(text, words):
    """Whether generated policy text contains any expected keyword/action."""
    return any(_keyword_present(text, word) for word in words)


def _max_examples_from_text(text):
    """Extract max_examples from a generated policy, if present."""
    import re
    match = re.search(r"max_examples\s*[:=]\s*(\d+)", text)
    return int(match.group(1)) if match else None


def evaluate_campaign_policy(component, _task_id):
    """Score a generated policy for adaptive recursive-opt campaign control."""
    scores, feedbacks = [], []
    for case in CAMPAIGN_POLICY_CASES:
        raw = component(case["diagnostics"])
        text = _policy_text(raw)
        selected_tasks = _mentioned_tasks(text, CAMPAIGN_TASKS)
        max_examples = _max_examples_from_text(text)
        action_score = 1.0 if _contains_any(text, case["actions"]) else 0.0
        task_score = 1.0 if not case["tasks"] else min(1.0, len(selected_tasks & case["tasks"]) / len(case["tasks"]))
        avoid_score = 1.0 if not (selected_tasks & case["avoid"]) else 0.0
        if max_examples is None:
            budget_score = 0.0
        else:
            lo, hi = case["max_examples"]
            budget_score = 1.0 if lo <= max_examples <= hi else 0.0
        reason_score = min(1.0, sum(1 for word in case["reasons"] if _keyword_present(text, word)) / 2.0)
        score = 0.30 * action_score + 0.25 * task_score + 0.20 * avoid_score + 0.15 * budget_score + 0.10 * reason_score
        scores.append(score)
        feedbacks.append(
            f"{case['name']}: score={score:.2f}; selected={sorted(selected_tasks)}; max_examples={max_examples}; "
            f"need_action={sorted(case['actions'])}; need_tasks={sorted(case['tasks'])}; avoid={sorted(case['avoid'])}; text={text[:200]!r}"
        )
    mean = statistics.mean(scores)
    return mean, " | ".join(feedbacks)


_BASELINES["campaign_policy"] = _baseline_campaign_policy


---
## Use Case 9 — Agentic Trace policy: tools + hints, not fixed tool lists

**Why:** fixed optimizer-side tools were mostly flat. The useful version is to learn a policy that selects optimizer tools *and* gives the optimizer a short purpose hint. This is the best current path toward Agentic Trace without changing core optimizer internals.

This improves the earlier UC5 tool-policy arm by adding reverse cases: saturated/low-spread campaigns should avoid expensive tools, while transfer/noisy cases should ask for retrieval or subset validation.

**Mode:** LIVE code-surface rewrite. The artifact is selector code that can be reused as an optimizer-tool policy.


In [9]:
# Use Case 9 — richer Agentic Trace tool policy with purpose hints.
def _baseline_agentic_trace_policy(self, signal):
    return "tools: note\nhint: observe the feedback"

AGENTIC_TRACE_CASES = [
    {"signal": "Need prior failures and family examples before proposing a prompt update.",
     "required": {"trace_search", "note"}, "hint_terms": {"prior", "failure", "family"}},
    {"signal": "Need validate a candidate on a small subset before accepting it.",
     "required": {"run_subset"}, "hint_terms": {"validate", "subset", "accept"}},
    {"signal": "Need inspect the saved artifact for syntax and current_code reuse.",
     "required": {"artifact_linter"}, "hint_terms": {"syntax", "artifact", "code"}},
    {"signal": "Task is saturated at 1.0 with zero gain; treat as control and avoid expensive tool calls.",
     "required": set(), "hint_terms": {"satur", "control", "avoid", "stop"}},
    {"signal": "Noisy transfer result: compare cold versus warm prior on held-out families before promoting.",
     "required": {"trace_search", "run_subset"}, "hint_terms": {"transfer", "holdout", "warm", "cold", "promot"}},
]


def evaluate_agentic_trace_policy(component, _task_id):
    """Score optimizer-tool selection plus the purpose hint for Agentic Trace."""
    scores, feedbacks = [], []
    for case in AGENTIC_TRACE_CASES:
        raw = component(case["signal"])
        text = _policy_text(raw)
        selected = parse_optimizer_tool_policy(raw, OPTIMIZER_TOOL_NAMES, max_tools=3)
        selected_set = set(selected)
        required = set(case["required"])
        expensive = selected_set - {"note"}
        if required:
            coverage = len(selected_set & required) / len(required)
            extras = len(selected_set - required - {"note"})
            tool_score = max(0.0, coverage - 0.15 * extras)
        else:
            tool_score = 1.0 if not expensive else max(0.0, 1.0 - 0.45 * len(expensive))
        hint_score = min(1.0, sum(1 for term in case["hint_terms"] if term.lower() in text) / 2.0)
        score = 0.70 * tool_score + 0.30 * hint_score
        scores.append(score)
        feedbacks.append(
            f"signal={case['signal']!r}; score={score:.2f}; selected={selected}; "
            f"required={sorted(required)}; hint_terms={sorted(case['hint_terms'])}; text={text[:180]!r}"
        )
    mean = statistics.mean(scores)
    return mean, " | ".join(feedbacks)


_BASELINES["agentic_trace_policy"] = _baseline_agentic_trace_policy


---
## Use Case 10 — Artifact promotion policy: promote/retest/reject generated solutions


In [10]:
# Use Case 10 — executable artifact-promotion policy.
# This version uses the generic GuardedDecisionEvaluator: the generated policy
# may return JSON, key-value text, or free text, but safety guards are scored
# lexicographically where promotion would be unsafe.
from opto.features.recursive_opt import ConfidenceGate, GuardedDecisionCase, GuardedDecisionEvaluator


def _baseline_promotion_policy(self, artifact_report):
    return "action: promote\nreason: best score"


PROMOTION_CONFIDENCE_GATE = ConfidenceGate(min_support=2, z=1.0, min_gain=0.0)

PROMOTION_CASES = [
    {"name": "validated_code_gain", "report": {"kind": "code", "mean_score": 1.0, "initial": 0.625,
      "std": 0.0, "n": 2, "artifact_version": 1, "syntax_ok": True, "saturated_control": False},
     "actions": {"promote"}, "required": {"code", "validated", "gain"}, "forbidden": {"reject", "control"}},
    {"name": "saturated_stride_control", "report": {"kind": "code", "mean_score": 1.0, "initial": 1.0,
      "std": 0.0, "n": 2, "artifact_version": 0, "syntax_ok": True, "saturated_control": True},
     "actions": {"control", "archive", "do_not_promote", "skip"}, "required": {"satur", "control"}, "forbidden": {"promote"}},
    {"name": "single_seed_noisy_config", "report": {"kind": "config", "mean_score": 0.16, "initial": 0.13,
      "std": 0.08, "n": 1, "artifact_version": 0, "syntax_ok": True, "saturated_control": False},
     "actions": {"retest", "hold", "probe"}, "required": {"config", "single", "retest"}, "forbidden": {"promote"}},
    {"name": "invalid_syntax_code", "report": {"kind": "code", "mean_score": -1.0, "initial": 0.4,
      "std": 0.0, "n": 2, "artifact_version": 1, "syntax_ok": False, "saturated_control": False},
     "actions": {"reject", "repair"}, "required": {"syntax", "reject"}, "forbidden": {"promote"}},
    {"name": "warm_prior_regression", "report": {"kind": "prior", "mean_score": -0.09, "initial": -0.01,
      "std": 0.002, "n": 2, "artifact_version": 0, "syntax_ok": True, "saturated_control": False},
     "actions": {"reject", "rollback", "cold", "do_not_promote"}, "required": {"regress", "rollback"}, "forbidden": {"promote"}},
]


def _promotion_guard_case(case):
    """Map a promotion report to one generic guarded-decision case."""
    report = case["report"]
    required = set(case["required"])
    if PROMOTION_CONFIDENCE_GATE.needs_retest(report):
        required.update({"single", "retest"})
    elif not PROMOTION_CONFIDENCE_GATE.clears_promotion(report) and report.get("kind") in {"prior", "config"}:
        required.update({"regress", "rollback"})
    return GuardedDecisionCase(
        name=case["name"],
        payload=report,
        allowed_actions=tuple(sorted(case["actions"])),
        required_terms=tuple(sorted(required)),
        forbidden_terms=tuple(sorted(case["forbidden"])),
        weights={"action": 0.45, "required": 0.35, "forbidden": 0.20},
        required_denominator=min(2, len(required)),
        hard_forbidden="promote" in case["forbidden"],
        forbidden_floor=0.0,
    )


PROMOTION_GUARDED_EVALUATOR = GuardedDecisionEvaluator(
    tuple(_promotion_guard_case(case) for case in PROMOTION_CASES)
)


def evaluate_promotion_policy(component, _task_id):
    """Score a generated artifact promotion/retest/reject policy with hard guards."""
    return PROMOTION_GUARDED_EVALUATOR(component, _task_id)


_BASELINES["promotion_policy"] = _baseline_promotion_policy


---
## Use Case 11 — Code-emitted Trace-Bench prompt artifact — FRONTIER

**Why:** UC2 showed that raw config/prompt optimization is slow and often weakly causal. This arm keeps the real Trace-Bench scoring path, but moves the optimized surface back to executable code: the learned function emits the `starting_artifact` prompt that Trace-Bench actually injects before scoring.

**What it proves if it works:** recursive_opt can learn reusable generator code for task artifacts, not only direct solvers or toy helper functions.

**Limit:** QASPER is intentionally slower/noisier, so use the five-seed summary before interpreting a frontier score gain as reliable.


In [11]:
# Use Case 11 — executable prompt-emitter scored through real Trace-Bench artifact injection.
def _qasper_prompt_emitter(self):
    """Weak seed: emit no prompt artifact, leaving the bundle default behavior."""
    return ""


_BASELINES["qasper_prompt_emitter"] = _qasper_prompt_emitter


In [ ]:
# Master roll-up: all experiments, best per use case, and previous-run artifact summaries.
ALL = {"UC1 component code": uc1, "UC2 setup/config": uc2, "UC3 capability": uc3,
       "UC4 family/transfer": uc4, "UC5 optimizer/tool": uc5, "UC6 trace feedback": uc6,
       "UC7 graph/suboptimizer": uc7, "UC8 campaign policy": uc8,
       "UC9 agentic trace policy": uc9, "UC10 promotion policy": uc10,
       "UC11 prompt emitter": uc11}

# Keep this summary cell rerunnable in an existing kernel: earlier cells may
# still hold older helper definitions, so derive display-only progress fields here.
def _summary_artifact_version_from_ref(ref):
    """Best-effort artifact version parsed from '<file>#family:kind:version:id'."""
    artifact_id = str(ref or "").rsplit("#", 1)[-1]
    parts = artifact_id.split(":")
    if len(parts) < 3:
        return None
    try:
        return int(parts[-2])
    except (TypeError, ValueError):
        return None


def _summary_artifact_version(result_or_row):
    """Return artifact lineage version from result metadata or artifact ref."""
    if not isinstance(result_or_row, dict):
        return None
    version = result_or_row.get("artifact_version")
    if version is not None:
        return version
    return _summary_artifact_version_from_ref(result_or_row.get("artifact_file"))


def _summary_best_step(result_or_row):
    """Return the optimizer level step where the best objective appeared."""
    if not isinstance(result_or_row, dict):
        return None
    step = result_or_row.get("best_step")
    if step is not None:
        return step
    progress = result_or_row.get("progress")
    if isinstance(progress, dict):
        return _best_step_from_progress(progress)
    return None


def _summary_fmt_int(value):
    """Format an optional integer-ish progress value for summary tables."""
    if value is None:
        return "-"
    try:
        return str(int(value))
    except (TypeError, ValueError):
        return str(value)


def _past_runs_table_with_progress(rows):
    """Render historical run rows with separate step and artifact-version columns."""
    head = "| run | use case | initial mean | final mean | best score | best step | artifact version | n memory dirs | best artifact file |\n|---|---|---:|---:|---:|---:|---:|---:|---|"
    lines = [head]
    for row in rows:
        lines.append(f"| {_md_cell(row['run'])} | {_md_cell(row['use_case'])} | {_fmt(row['initial_mean'])} | "
                     f"{_fmt(row['final_mean'])} | {_fmt(row['best_score'])} | {_summary_fmt_int(_summary_best_step(row))} | {_summary_fmt_int(_summary_artifact_version(row))} | {row['n_dirs']} | "
                     f"{_md_code(row['artifact_file'])} |")
    return "\n".join(lines)


def _past_experiments_table_with_progress(rows, limit=None):
    """Render historical experiment rows with separate step and artifact-version columns."""
    head = "| run | use case | experiment | initial | best score | best step | artifact version | best artifact file |\n|---|---|---|---:|---:|---:|---:|---|"
    lines = [head]
    ordered = sorted(rows, key=lambda r: (r["run"], r["use_case"], r["experiment"]))
    selected = ordered if limit is None else ordered[-limit:]
    for row in selected:
        lines.append(f"| {_md_cell(row['run'])} | {_md_cell(row['use_case'])} | {_md_cell(row['experiment'])} | "
                     f"{_fmt(row['initial'])} | {_fmt(row['best_score'])} | {_summary_fmt_int(_summary_best_step(row))} | {_summary_fmt_int(_summary_artifact_version(row))} | "
                     f"{_md_code(row['artifact_file'])} |")
    if limit is not None and len(ordered) > limit:
        lines.append(f"| ... | ... | {len(ordered)-limit} older rows omitted | - | - | - | - | - |")
    return "\n".join(lines)



def _sanitize_export_name(value):
    """Return a stable filesystem-safe name for exported artifact files."""
    import re
    text = re.sub(r"[^a-zA-Z0-9_.-]+", "_", str(value).strip().lower())
    return text.strip("._")[:80] or "artifact"


def _record_from_artifact_ref(ref):
    """Load the artifact JSONL record referenced by a summary table cell."""
    if not ref or ref == "-":
        return None
    file_part, sep, artifact_id = str(ref).partition("#")
    path = Path(file_part)
    records = _read_jsonl(path)
    if not records:
        return None
    if sep:
        for record in records:
            if record.get("artifact_id") == artifact_id:
                return record
    scored = [(score[0], record) for record in records if (score := _finite([record.get("score")]))]
    return max(scored, key=lambda item: item[0])[1] if scored else records[-1]


def _artifact_suffix(kind, content):
    """Choose a reusable extension by artifact kind/content."""
    if kind == "code":
        return ".py"
    if kind == "graph":
        return ".json"
    return ".txt"


def export_best_artifacts(all_results):
    """Materialize best per-use-case artifacts as standalone files plus an index."""
    out_dir = OUTPUT_ROOT / "best_artifacts"
    out_dir.mkdir(parents=True, exist_ok=True)
    index = []
    for number, (use_case, rows) in enumerate(all_results.items(), start=1):
        best = best_of(rows)
        if best is None:
            continue
        label, result = best
        record = _record_from_artifact_ref(result.get("artifact_file"))
        content = record.get("content") if isinstance(record, dict) else result.get("artifact")
        if content is None:
            continue
        kind = record.get("kind") if isinstance(record, dict) else "artifact"
        suffix = _artifact_suffix(kind, content)
        stem = f"{number:02d}_{_sanitize_export_name(use_case)}__{_sanitize_export_name(label)}"
        path = out_dir / f"{stem}{suffix}"
        if isinstance(content, (dict, list)):
            path.write_text(json.dumps(content, indent=2, sort_keys=True) + "\n")
        else:
            path.write_text(str(content).rstrip() + "\n")
        mean = _result_mean(result)
        item = {
            "use_case": use_case,
            "experiment": label,
            "kind": kind,
            "score": record.get("score") if isinstance(record, dict) else mean,
            "initial": result.get("initial"),
            "mean_score": mean,
            "artifact_ref": result.get("artifact_file"),
            "export_file": str(path),
            "spec_file": result.get("spec_file"),
        }
        index.append(item)
    (out_dir / "index.json").write_text(json.dumps(index, indent=2, sort_keys=True, default=str) + "\n")
    readme_lines = ["# Best recursive_opt artifacts", "", "Generated from the current notebook run.", ""]
    for item in index:
        readme_lines.append(f"- {item['use_case']} / {item['experiment']} -> `{item['export_file']}` (kind={item['kind']}, score={_fmt(item['score'])})")
    (out_dir / "README.md").write_text("\n".join(readme_lines) + "\n")
    return index


def _artifact_exports_table(index):
    """Render exported standalone artifacts for reuse."""
    head = "| use case | experiment | kind | score | export file | source artifact |\n|---|---|---|---:|---|---|"
    lines = [head]
    for item in index:
        lines.append(f"| {_md_cell(item['use_case'])} | {_md_cell(item['experiment'])} | {_md_cell(item['kind'])} | "
                     f"{_fmt(item['score'])} | {_md_code(item['export_file'])} | {_md_code(item['artifact_ref'])} |")
    return "\n".join(lines)

flat = ["| use case | experiment | initial | mean score | delta | std | n | wall_s | best step | artifact version | best artifact file | spec file | notes | best? |",
        "|---|---|---:|---:|---:|---:|---:|---:|---:|---:|---|---|---|---|"]
for uc, data in ALL.items():
    best = best_of(data)
    best_label = best[0] if best else None
    for label, result in data:
        scores = _finite(result.get("scores", []))
        mean = statistics.mean(scores) if scores else None
        std = statistics.pstdev(scores) if len(scores) > 1 else None
        delta = (mean - result["initial"]) if mean is not None and result.get("initial") is not None else None
        flat.append(f"| {_md_cell(uc)} | {_md_cell(label)} | {_fmt(result.get('initial'))} | {_fmt(mean)} | {_fmt(delta)} | "
                    f"{_fmt(std)} | {len(scores)} | {_fmt(result.get('wall_s'))} | {_summary_fmt_int(_summary_best_step(result))} | {_summary_fmt_int(_summary_artifact_version(result))} | "
                    f"{_md_code(result.get('artifact_file') or '-')} | {_md_code(result.get('spec_file') or '-')} | "
                    f"{_md_cell(_notes_for_result(result))} | {'yes' if label == best_label else ''} |")

_display_markdown("### All current-run results\n" + "\n".join(flat))

best_rows = ["| use case | best experiment | initial | mean score | delta | n | wall_s | best step | artifact version | best artifact file | spec file |",
             "|---|---|---:|---:|---:|---:|---:|---:|---:|---|---|"]
for uc, data in ALL.items():
    b = best_of(data)
    if b is None:
        best_rows.append(f"| {_md_cell(uc)} | (offline / no live result) | - | - | - | 0 | - | - | - | - | - |")
    else:
        label, result = b
        mean = _result_mean(result)
        delta = (mean - result["initial"]) if mean is not None and result.get("initial") is not None else None
        best_rows.append(f"| {_md_cell(uc)} | {_md_cell(label)} | {_fmt(result.get('initial'))} | {_fmt(mean)} | {_fmt(delta)} | "
                         f"{len(_finite(result.get('scores', [])))} | {_fmt(result.get('wall_s'))} | {_summary_fmt_int(_summary_best_step(result))} | {_summary_fmt_int(_summary_artifact_version(result))} | "
                         f"{_md_code(result.get('artifact_file') or '-')} | {_md_code(result.get('spec_file') or '-')} |")
_display_markdown("### Best result per use case\n" + "\n".join(best_rows))

exports = export_best_artifacts(ALL)
_display_markdown("### Standalone best-artifact exports\n" + _artifact_exports_table(exports))

past = summarize_past_runs(OUTPUT_ROOT.parent)
if past:
    _display_markdown("### Historical persisted-artifact summary\n" + _past_runs_table_with_progress(past))
    detailed = summarize_past_experiments(OUTPUT_ROOT.parent)
    _display_markdown("### Historical persisted-artifact detail (all past experiments)\n" + _past_experiments_table_with_progress(detailed))
else:
    _display_markdown("### Historical persisted-artifact summary\nNo prior output folders found.")

_display_markdown("**Interpretation guardrails:** UC1/UC5 code-helper scores are validator evidence and can saturate; "
                 "UC2/UC6 are real Trace-Bench prompt/config scores over the configured examples; "
                 "UC1 now includes a real BBEH direct-code arm to avoid relying only on toy validators; "
                 "UC3 remains a prompt-capability surface on GSM8K; UC4 transfer is only meaningful when warm beats cold by more than run noise. "
                 "For config arms, inspect whether the saved config actually changed; if `starting_artifact` is blank/unchanged, treat the gain as benchmark/trace variance or trace-condition evidence, not as a learned prompt. "
                 "UC8/UC9/UC10 are structured policy-code experiments: they test meta-campaign decisions, Agentic Trace tool/hint selection, and artifact promotion without changing core optimizer internals. "
                 "UC11 is the frontier bridge from weak config tuning to executable prompt-emitter code scored by real Trace-Bench artifact injection. "
                 "Reusable artifacts are in each listed `artifacts.jsonl` under the `content` field and are also materialized under `best_artifacts/` with an `index.json`; adjacent `spec.json`, `component_spec.json`, or `graph_spec.json` files record how each solution was produced. Code arms save Python code, config arms save config text, learned policy arms save selector/controller code, and optimizer-tool-calling arms save the chosen config plus evidence hooks.")


---

## Use Case 12 - Six promoted recursive_opt primitives

This section validates the six promoted primitives from the new recursive_opt patch without replacing the live UC1-UC11 benchmark results above. It checks three things per primitive: API/spec usability, causal plumbing, and whether the generated artifacts/results are persisted under the common notebook output root.

Interpretation boundary: the budget/seeds/numeric/search-policy checks are deterministic API and causality proofs. The benchmark performance evidence remains the live Trace-Bench runs in UC1-UC11 and any live config checks executed here with the registered Trace-Bench adapter.


In [ ]:

# Use Case 12 - six promoted primitives from the new recursive_opt patch.
from opto.optimizers.optimizer import Optimizer
from opto.trace.nodes import node as trace_node
from opto.features.recursive_opt import (
    run_spec as recursive_run_spec,
    run_spec_repeated,
    RepeatedResult,
    seed_everything,
    make_level_spec,
    MemoryLite,
    route_optimizers,
    OptunaOptimizer,
    LeastSquaresOptimizer,
    field_search_space,
    is_numeric_field,
    run_search_policy,
    make_search_policy_tool,
    make_search_policy_evaluator,
)
from opto.features.recursive_opt.budget import (
    make_budget as make_recursive_budget,
    budget_to_spec_dict,
    current_budget,
    reset_budget,
)
from opto.features.recursive_opt import spec as recursive_spec
from opto.features.recursive_opt import tracebench as TB
from opto.features.recursive_opt.effects import effects_for
from opto.features.recursive_opt.levels import LevelConfig
from opto.features.recursive_opt.tracebench import _summarize_feedbacks

UC12_ROWS = []

class NotebookNoLLMOptimizer(Optimizer):
    """No-op optimizer for API checks that should not spend LLM calls."""

    def __init__(self, parameters, **kwargs):
        super().__init__(parameters)
        self.steps = 0

    def step(self, *args, **kwargs):
        self.steps += 1
        return {}

    def zero_feedback(self):
        return None

    def backward(self, *args, **kwargs):
        return None


def record_uc12(item, variant, status, metric=None, artifact=None, notes=""):
    """Append one UC12 validation row with consistent fields."""
    UC12_ROWS.append({
        "item": item,
        "variant": variant,
        "status": status,
        "metric": metric,
        "artifact": artifact,
        "notes": notes,
    })


def uc12_table(rows):
    """Render UC12 validation rows as markdown."""
    head = "| item | variant | status | metric | artifact/result | notes |\n|---|---|---|---:|---|---|"
    lines = [head]
    for row in rows:
        lines.append(
            f"| {_md_cell(row['item'])} | {_md_cell(row['variant'])} | {_md_cell(row['status'])} | "
            f"{_md_cell(_fmt(row.get('metric')) if isinstance(row.get('metric'), (int, float)) else row.get('metric'))} | "
            f"{_md_code(row.get('artifact')) if row.get('artifact') else '-'} | {_md_cell(row.get('notes'))} |"
        )
    return "\n".join(lines)


def deterministic_capability_eval(capability_callable, _family):
    """Tiny evaluator used to exercise run_spec without an LLM optimizer."""
    text = str(capability_callable(task="uc12").get("answer", ""))
    score = 1.0 if text else 0.0
    return {"accuracy": score}, f"deterministic capability score={score}", score


In [ ]:

# Item 4 + Item 3: budget dict promotion and true multi-seed execution.
budget_dict = {"wall_time_s": 1500, "optimizer_llm_calls": 8,
               "eval_llm_calls": 24, "candidates": 16,
               "on_exceed": "raise"}
roundtrip_ok = budget_to_spec_dict(make_recursive_budget(budget_dict)) == budget_dict
record_uc12("Item 4 budget", "lossless make_budget/to_spec_dict", "pass" if roundtrip_ok else "fail",
            metric=1.0 if roundtrip_ok else 0.0, notes="Dict, object, and method forms share one mapping.")

spec_with_budget = {
    "memory_root": memory_path("mem_uc12_budget_override"),
    "budget": {"candidates": 99},
    "levels": [make_level_spec(id="budget_probe", surface="capability",
                                seed="probe", evaluator=deterministic_capability_eval,
                                iterations=1)],
}
before_budget = dict(spec_with_budget["budget"])
reset_budget()
out_budget = recursive_run_spec(spec_with_budget, optimizer=NotebookNoLLMOptimizer,
                                budget={"candidates": 7, "on_exceed": "return_best"})
override_ok = current_budget().max_candidates == 7 and spec_with_budget["budget"] == before_budget
record_uc12("Item 4 budget", "run_spec budget override isolation", "pass" if override_ok else "fail",
            metric=current_budget().max_candidates,
            artifact=out_budget["results"]["budget_probe"].get("artifact_id"),
            notes="Override applies to the run and leaves spec['budget'] unmutated.")

import random
seed_everything(123)
first_random = [round(random.random(), 6) for _ in range(3)]
seed_everything(123)
second_random = [round(random.random(), 6) for _ in range(3)]
record_uc12("Item 3 seeds", "seed_everything controls RNG", "pass" if first_random == second_random else "fail",
            metric=1.0 if first_random == second_random else 0.0,
            notes=f"random sequence={first_random}")

seed_spec = {
    "memory_root": memory_path("mem_uc12_seeded"),
    "budget": {"candidates": 20, "on_exceed": "return_best"},
    "levels": [make_level_spec(id="seeded", surface="capability",
                                seed="seeded policy", evaluator=deterministic_capability_eval,
                                iterations=1)],
}
seeded = recursive_run_spec(seed_spec, seeds=[0, 1, 2], optimizer=NotebookNoLLMOptimizer)
seed_rr = seeded["seeded"]
record_uc12("Item 3 seeds", "run_spec(seeds=) returns RepeatedResult", "pass" if isinstance(seed_rr, RepeatedResult) else "fail",
            metric=seed_rr.mean(), artifact=memory_path("mem_uc12_seeded_seed0"),
            notes=f"n_valid={seed_rr.n_valid()}, errors={len(seed_rr.errors)}")


In [ ]:

# Item 1: batch_design and credit_horizon are active consumers in the real adapter.
try:
    active_adapter = TB.TraceBenchTaskAdapter.from_config(
        tracebench_block(max_examples=min(6, MAX_EXAMPLES), inner_steps=1, timeout_seconds=25, eval_kwargs={"n_train": 6, "n_val": 1})
    )
    TB.register_task_adapter(active_adapter)
    contract = effects_for(active_adapter)
    active_ok = bool(contract["batch_design"].active and contract["credit_horizon"].active)
    record_uc12("Item 1 active fields", "adapter effect contract", "pass" if active_ok else "fail",
                metric=1.0 if active_ok else 0.0,
                notes=f"batch_design={contract['batch_design'].effects}; credit_horizon={contract['credit_horizon'].effects}")

    task_id = "internal:multiobjective_bbeh"
    bundle = active_adapter._load_bundle(task_id, fresh=True)
    train_dataset = bundle["train_dataset"]
    inputs = list(train_dataset.get("inputs") or [])[: min(6, active_adapter.max_examples)]
    infos = list(train_dataset.get("infos") or train_dataset.get("info") or [None] * len(inputs))[: len(inputs)]
    batch_orders = {}
    for design in ["random", "failure_balanced", "curriculum", "diversity"]:
        ordered_inputs, _ordered_infos = active_adapter._order_by_batch_design(
            inputs, infos, LevelConfig(batch_design=design, batch_size=4)
        )
        batch_orders[design] = [len(str(value)) for value in ordered_inputs]
    order_changed = len({tuple(order) for order in batch_orders.values()}) > 1
    record_uc12("Item 1 active fields", "batch_design changes inner-training batch order",
                "pass" if order_changed else "flat",
                metric=len({tuple(order) for order in batch_orders.values()}),
                notes=f"orders_by_input_length={batch_orders}; this proves the consumer path before score interpretation.")
except Exception as exc:
    record_uc12("Item 1 active fields", "real Trace-Bench batch_design probe", "fail",
                notes=_one_line_error(exc))

feedbacks = [f"feedback {i}: failure mode {i % 3}" for i in range(5)]
horizon_lengths = {h: len(_summarize_feedbacks(feedbacks, h)) for h in ["truncated", "episode", "step", "full"]}
horizon_ok = len(set(horizon_lengths.values())) > 1
record_uc12("Item 1 active fields", "credit_horizon changes optimizer-visible feedback", "pass" if horizon_ok else "fail",
            metric=max(horizon_lengths.values()) - min(horizon_lengths.values()),
            notes=f"summary lengths={horizon_lengths}")


In [ ]:

# Item 2: non-generative numeric optimizers and routing policy.
plan = route_optimizers(["starting_artifact", "batch_design", "batch_size"],
                        policy={"order": "numeric_then_text", "numeric_optimizer": "optuna"})
routing_ok = plan["numeric_fields"] == ["batch_design", "batch_size"] and plan["text_fields"] == ["starting_artifact"]
record_uc12("Item 2 numeric routing", "mixed target routing", "pass" if routing_ok else "fail",
            metric=len(plan["numeric_fields"]), notes=str(plan))

def numeric_eval(assignment):
    score = 0.6 if assignment.get("batch_design") == "failure_balanced" else 0.0
    score += 0.4 * (assignment.get("batch_size", 1) / 8.0)
    return score

opt = OptunaOptimizer([trace_node("x", trainable=True, name="uc12_cfg")],
                      evaluate=numeric_eval,
                      space=field_search_space(["batch_design", "batch_size"]),
                      max_trials=30)
best_numeric = opt.step()
best_numeric_score = max(score for _assignment, score in opt.history)
record_uc12("Item 2 numeric routing", "OptunaOptimizer/fallback learns categorical+int optimum",
            "pass" if best_numeric == {"batch_design": "failure_balanced", "batch_size": 8} else "fail",
            metric=best_numeric_score, artifact=str(best_numeric),
            notes=f"history_len={len(opt.history)}; optuna is optional, fallback is deterministic.")

ls = LeastSquaresOptimizer([trace_node("x", trainable=True, name="uc12_ls")],
                           evaluate=lambda a: a.get("batch_size", 1) / 8.0,
                           space=field_search_space(["batch_size"]),
                           max_trials=15, target=1.0)
ls.step()
record_uc12("Item 2 numeric routing", "LeastSquaresOptimizer handles integer numeric field",
            "pass" if ls.best_assignment and ls.best_assignment.get("batch_size") == 8 else "fail",
            metric=ls.best_assignment.get("batch_size") if ls.best_assignment else None,
            notes="Continuous solve rounded back into the integer field domain.")

# Item 5: active family/prior defaults and initial_knowledge as optimizer docs.
try:
    TB.register_task_adapter(TB.TraceBenchTaskAdapter.from_config(tracebench_block(max_examples=2, inner_steps=1, timeout_seconds=20)))
    families = {"combo": ["llm4ad:online_bin_packing_local"],
                "math": ["internal:multiobjective_gsm8k"]}
    mem_prior = MemoryLite(root=memory_path("mem_uc12_prior_defaults"))
    o2 = recursive_spec.compile_level(make_level_spec(id="o2_default", surface="family_policy", family="*"), mem_prior, families)
    o3 = recursive_spec.compile_level(make_level_spec(id="o3_default", surface="prior", family="*"), mem_prior, families)
    defaults_ok = ("memory_policy" not in o2._fields and "starting_artifact" in o2._fields
                   and "batch_design" in o3._fields)
    record_uc12("Item 5 priors", "active default policy/prior fields", "pass" if defaults_ok else "fail",
                metric=len(o2._fields), notes=f"o2_fields={o2._fields}; o3_fields={o3._fields}")

    doc_adapter = TB.TraceBenchTaskAdapter.from_config(tracebench_block(max_examples=1, inner_steps=0, timeout_seconds=10))
    doc_node = trace_node("clean artifact", trainable=True, name="uc12_artifact")
    doc_adapter._apply_starting_artifact({"param": doc_node},
                                         LevelConfig(initial_knowledge="Prefer verification before final answer."))
    description = "\n".join(str(value) for value in [
        getattr(doc_node, "description", ""),
        getattr(doc_node, "_description", ""),
    ] if value)
    docs_ok = "prefer verification" in description.lower() and str(doc_node.data) == "clean artifact"
    record_uc12("Item 5 priors", "initial_knowledge reaches optimizer docs", "pass" if docs_ok else "fail",
                metric=1.0 if docs_ok else 0.0,
                notes="Artifact text stays clean; family prior is in the trainable node description.")
except Exception as exc:
    record_uc12("Item 5 priors", "prior/default validation", "fail", notes=_one_line_error(exc))

# Item 6: learnable active-search / lessons-learnt tool.
mem_search = MemoryLite(root=memory_path("mem_uc12_search_policy"))
for i, fb in enumerate([
    "parse failures disappear when examples include expected output format",
    "timeouts improve after preferring short candidate programs",
    "arithmetic answers need final verification",
    "parse failures recur when prompt omits JSON schema",
]):
    mem_search.record(level="O1", cfg={"i": i}, family="codegen", score=0.1 * i, feedback=fb)

recent_lesson = run_search_policy({"k": 2, "strategy": "recent", "template": "Lesson: {lessons}"}, mem_search, family="codegen")
diverse_lesson = run_search_policy({"k": 2, "strategy": "diverse", "template": "Lesson: {lessons}"}, mem_search, family="codegen")
record_uc12("Item 6 search policy", "policy changes retrieved lesson", "pass" if recent_lesson != diverse_lesson else "flat",
            metric=abs(len(recent_lesson) - len(diverse_lesson)),
            notes=f"recent='{recent_lesson[:60]}'; diverse='{diverse_lesson[:60]}'")

tool = make_search_policy_tool({"k": 3, "strategy": "all", "template": "Avoid: {lessons}"}, mem_search, family="codegen")
tool_lesson = tool("optimizer feedback")
base_eval = lambda prior_text: 0.9 if "parse" in str(prior_text).lower() else 0.5
evaluator = make_search_policy_evaluator(mem_search, base_eval, family="codegen")
lift, lift_feedback = evaluator({"k": 3, "strategy": "all", "template": "Avoid: {lessons}"}, "codegen")
record_uc12("Item 6 search policy", "tool callable and evaluator lift", "pass" if lift > 0 and tool_lesson else "fail",
            metric=lift, artifact=memory_path("mem_uc12_search_policy"),
            notes=lift_feedback)


In [ ]:

# UC12 summary: persisted under the same common output root.
uc12_path = OUTPUT_ROOT / "uc12_six_promotions.json"
uc12_path.write_text(json.dumps(UC12_ROWS, indent=2, sort_keys=True, default=str) + "\n")
uc12_passes = sum(1 for row in UC12_ROWS if str(row.get("status", "")).lower() in {"pass", "passed", "ok"})
uc12_total = len(UC12_ROWS)
uc12_pass_rate = uc12_passes / uc12_total if uc12_total else 0.0
uc12 = [("six promoted primitives validation", {
    "scores": [uc12_pass_rate],
    "initial": 0.0,
    "wall_s": None,
    "artifact": str(uc12_path),
    "artifact_id": None,
    "artifact_file": str(uc12_path),
    "best_step": None,
    "artifact_version": None,
    "progress": None,
    "spec_file": None,
    "errors": [],
    "dry": False,
    "notes": f"validation pass-rate over {uc12_total} promoted primitive checks; not a benchmark score",
})]
_display_markdown("### UC12 - six promoted recursive_opt primitives\n" + uc12_table(UC12_ROWS))
print("UC12 results saved to", uc12_path)


## Use Case 13 - numeric config optimizer head-to-head

UC13 isolates the new non-generative numeric optimizer path. In offline mode it
runs a small deterministic causal preflight through the same `MetaLevel` and
`optimize_config_numeric` bridge; that row proves wiring, not benchmark quality.
In live mode it adds the real Trace-Bench head-to-head under a tight budget:
`max_examples=6`, `inner_steps=2`, and `optimizer_llm_calls=8`.


In [11]:
# Use Case 13 - numeric optimizer vs LLM config search on active fields.
from opto.features.recursive_opt import optimize_config_numeric
from opto.features.recursive_opt import spec as recursive_spec
from opto.features.recursive_opt import tracebench as TB
from opto.features.recursive_opt.levels import LevelConfig, MetaLevel

UC13_TASK_OVERRIDE = os.environ.get("RECURSIVE_OPT_UC13_TASK")
UC13_CANDIDATE_TASKS = [
    task.strip()
    for task in os.environ.get(
        "RECURSIVE_OPT_UC13_TASK_CANDIDATES",
        "hf:qasper,internal:multiobjective_gsm8k",
    ).split(",")
    if task.strip()
]
UC13_TASK = UC13_TASK_OVERRIDE or (UC13_CANDIDATE_TASKS[0] if UC13_CANDIDATE_TASKS else "hf:qasper")
UC13_FIELDS = list(CAUSAL_NUMERIC_TARGETS)
UC13_MAX_EXAMPLES = int(os.environ.get("RECURSIVE_OPT_UC13_MAX_EXAMPLES", "6"))
UC13_PILOT_MAX_EXAMPLES = int(os.environ.get("RECURSIVE_OPT_UC13_PILOT_MAX_EXAMPLES", str(min(4, UC13_MAX_EXAMPLES))))
UC13_INNER_STEPS = int(os.environ.get("RECURSIVE_OPT_UC13_INNER_STEPS", "2"))
UC13_TRIALS = int(os.environ.get("RECURSIVE_OPT_UC13_TRIALS", "12" if LIVE else "24"))
# Keep the live comparison budget tight, but let the offline causal preflight
# cover enough categorical/int combinations to prove the numeric optimizer path.
UC13_OFFLINE_TRIALS = int(os.environ.get("RECURSIVE_OPT_UC13_OFFLINE_TRIALS", "24"))
UC13_BUDGET = {**budget_block(), "optimizer_llm_calls": 8,
               "eval_llm_calls": min(MAX_EVAL_CALLS, 48),
               "candidates": min(MAX_CANDIDATES, 8)}
UC13_PILOT_ASSIGNMENTS = [
    {"batch_design": "random", "batch_size": 2},
    {"batch_design": "failure_balanced", "batch_size": 8},
    {"batch_design": "diversity", "batch_size": 2},
    {"batch_design": "curriculum", "batch_size": 4},
]


def uc13_tracebench_block(max_examples=None):
    """Trace-Bench bounds for the live UC13 head-to-head."""
    return tracebench_block(max_examples=max_examples or UC13_MAX_EXAMPLES,
                            inner_steps=UC13_INNER_STEPS)


def _uc13_safe_name(task):
    """Filesystem-safe task label for UC13 pilot subdirectories."""
    return str(task).replace(":", "_").replace("/", "_").replace(".", "_")


def _uc13_config_from_assignment(assignment):
    """Build a LevelConfig from one numeric/categorical assignment."""
    cfg = LevelConfig()
    for field, value in assignment.items():
        setattr(cfg, field, value)
    return cfg


def _uc13_result(root, label, initial, best_assignment, best_score, history, wall_s,
                 spec_file=None, notes=""):
    """Build a table-compatible UC13 result and persist the learning curve."""
    eval_calls = len(history)
    curve = [round(float(score), 3) for _assignment, score in history]
    payload = {
        "label": label,
        "initial": initial,
        "best_assignment": best_assignment,
        "best_score": best_score,
        "history": history,
        "curve": curve,
        "task": UC13_TASK,
        "fields": UC13_FIELDS,
        "live": LIVE,
        "wall_s": round(float(wall_s), 1),
        "eval_calls": eval_calls,
    }
    artifact_file = write_experiment_json(root, "uc13_numeric_result.json", payload)
    artifact = json.dumps({
        "best_assignment": best_assignment,
        "best_score": best_score,
        "curve": curve,
        "history_len": eval_calls,
        "task": UC13_TASK,
    }, indent=2, sort_keys=True, default=str)
    return {"scores": [float(best_score)], "initial": float(initial),
            "wall_s": round(float(wall_s), 1), "eval_calls": eval_calls,
            "artifact": artifact,
            "artifact_id": "uc13:numeric:best", "artifact_file": artifact_file,
            "best_step": (max(range(len(curve)), key=lambda index: curve[index]) if curve else None),
            "artifact_version": None, "progress": {"history": history},
            "spec_file": spec_file, "errors": [], "dry": False,
            "notes": notes or "zero LLM proposal calls; each trial still runs the real inner evaluator"}


def run_uc13_offline_preflight():
    """Run the causal numeric bridge without external services."""
    root = memory_path("mem_uc13_offline_numeric")
    mem = MemoryLite(root=root)

    def offline_runner(cfg, _task):
        design_score = {"failure_balanced": 0.55, "diversity": 0.35,
                        "curriculum": 0.20, "random": 0.05}.get(cfg.batch_design, 0.0)
        batch_score = min(max(float(cfg.batch_size), 1.0), 8.0) / 8.0 * 0.35
        score = design_score + batch_score + 0.10
        return score, f"causal_preflight design={cfg.batch_design} batch_size={cfg.batch_size} score={score:.3f}"

    level = MetaLevel(cfg=LevelConfig(), inner_runner=offline_runner,
                      trainable_fields=tuple(UC13_FIELDS), memory=mem)
    initial, _ = offline_runner(LevelConfig(), "offline_causal_preflight")
    t0 = time.time()
    best, best_score, history = optimize_config_numeric(
        level, "offline_causal_preflight", UC13_FIELDS,
        max_trials=UC13_OFFLINE_TRIALS,
        space=numeric_search_space(UC13_FIELDS, CAUSAL_NUMERIC_CONSTRAINTS))
    return _uc13_result(root, "offline numeric causal preflight", initial, best,
                        best_score, history, time.time() - t0)


def uc13_live_numeric_spec(root, task=None, max_examples=None):
    """Spec used to compile the real Trace-Bench numeric-only config level."""
    task_id = task or UC13_TASK
    return {"families": {"uc13": [task_id]}, "memory_root": root,
            "budget": dict(UC13_BUDGET), "tracebench": uc13_tracebench_block(max_examples=max_examples),
            "scoring": {"clip": [-1.0, 1.0]},
            "levels": [make_level_spec(
                id="uc13_numeric", surface="config", family="uc13", task=task_id,
                targets=UC13_FIELDS, constraints=CAUSAL_NUMERIC_CONSTRAINTS,
                fixed={"optimizer": "OptoPrimeV2", "trainer": "PrioritySearch",
                       "trace_type": "internal", "credit_horizon": "step"},
                iterations=RUN_ITERATIONS)]}


def run_uc13_task_signal_pilot():
    """Select a live task where active numeric fields measurably affect score."""
    if not LIVE:
        return UC13_TASK, mark_control({
            "scores": [], "initial": None, "wall_s": None, "eval_calls": 0,
            "artifact": "(offline: UC13 task pilot skipped)",
            "artifact_id": None, "artifact_file": None, "spec_file": None,
            "errors": [], "dry": True,
        }, "offline task pilot skipped")
    if UC13_TASK_OVERRIDE:
        return UC13_TASK_OVERRIDE, mark_control({
            "scores": [0.0], "initial": 0.0, "wall_s": 0.0, "eval_calls": 0,
            "artifact": f"task override: {UC13_TASK_OVERRIDE}",
            "artifact_id": None, "artifact_file": None, "spec_file": None,
            "errors": [], "dry": False,
        }, "task override supplied; spread pilot skipped")

    root = Path(memory_path("mem_uc13_task_signal_pilot"))
    started = time.time()
    task_rows = []
    errors = []
    for task_id in UC13_CANDIDATE_TASKS:
        try:
            task_root = root / _uc13_safe_name(task_id)
            spec = uc13_live_numeric_spec(str(task_root), task=task_id,
                                          max_examples=UC13_PILOT_MAX_EXAMPLES)
            spec_file = write_experiment_json(task_root, "spec.json", spec)
            TB.configure_tracebench_adapter(spec["tracebench"], require=True)
            mem = MemoryLite(root=str(task_root))
            level = recursive_spec.compile_level(spec["levels"][0], mem, spec["families"], spec.get("scoring"))
            for assignment in UC13_PILOT_ASSIGNMENTS:
                score, _feedback = level._inner_runner(_uc13_config_from_assignment(assignment), task_id)
                task_rows.append({
                    "task": task_id,
                    "assignment": dict(assignment),
                    "score": float(score),
                    "spec_file": spec_file,
                })
        except Exception as exc:
            errors.append(f"{task_id}: {_one_line_error(exc)}")

    summaries = []
    for task_id in UC13_CANDIDATE_TASKS:
        scores = [row["score"] for row in task_rows if row["task"] == task_id]
        if not scores:
            continue
        summaries.append({
            "task": task_id,
            "min": min(scores),
            "max": max(scores),
            "spread": max(scores) - min(scores),
            "mean": statistics.mean(scores),
            "n": len(scores),
        })
    selected = max(summaries, key=lambda row: (row["spread"], row["max"])) if summaries else {
        "task": UC13_TASK,
        "min": 0.0,
        "max": 0.0,
        "spread": 0.0,
        "mean": 0.0,
        "n": 0,
    }
    payload = {
        "selected_task": selected["task"],
        "summaries": summaries,
        "rows": task_rows,
        "errors": errors,
        "assignments": UC13_PILOT_ASSIGNMENTS,
        "max_examples": UC13_PILOT_MAX_EXAMPLES,
    }
    artifact_file = write_experiment_json(root, "uc13_task_signal_pilot.json", payload)
    result = {
        "scores": [float(selected["max"])],
        "initial": float(selected["min"]),
        "wall_s": round(time.time() - started, 1),
        "eval_calls": len(task_rows),
        "artifact": json.dumps(payload, indent=2, sort_keys=True, default=str),
        "artifact_id": "uc13:task_signal:pilot",
        "artifact_file": artifact_file,
        "best_step": None,
        "artifact_version": None,
        "progress": {"rows": task_rows},
        "spec_file": None,
        "errors": errors,
        "dry": False,
        "notes": f"selected {selected['task']} by spread={selected['spread']:.3f}; task-selection pilot, not an optimizer result",
    }
    return selected["task"], mark_control(result, "task-selection pilot; not optimizer evidence")


def run_uc13_live_numeric():
    """Run Optuna-style numeric search through the real Trace-Bench inner runner."""
    root = memory_path("mem_uc13_live_numeric")
    spec = uc13_live_numeric_spec(root)
    spec_file = write_experiment_json(root, "spec.json", spec)
    TB.configure_tracebench_adapter(spec["tracebench"], require=True)
    mem = MemoryLite(root=root)
    level = recursive_spec.compile_level(spec["levels"][0], mem, spec["families"], spec.get("scoring"))
    initial, _ = level._inner_runner(LevelConfig(), UC13_TASK)
    reset_standard_budget()
    t0 = time.time()
    best, best_score, history = optimize_config_numeric(
        level, UC13_TASK, UC13_FIELDS, max_trials=UC13_TRIALS,
        space=numeric_search_space(UC13_FIELDS, CAUSAL_NUMERIC_CONSTRAINTS))
    return _uc13_result(root, "live numeric config search", initial, best,
                        best_score, history, time.time() - t0, spec_file=spec_file,
                        notes=f"selected_task={UC13_TASK}; zero LLM proposal calls; eval_trials={len(history)}")


def uc13_live_llm_spec():
    """LLM optimizer arm over the same active numeric fields and bounds."""
    return config_spec(UC13_FIELDS, numeric_constraints=CAUSAL_NUMERIC_CONSTRAINTS,
                       memory_root="./mem_uc13_live_llm", task=UC13_TASK,
                       family_name="uc13", max_examples=UC13_MAX_EXAMPLES,
                       inner_steps=UC13_INNER_STEPS, budget=UC13_BUDGET)


uc13 = [("offline numeric causal preflight", run_uc13_offline_preflight())]
if LIVE:
    UC13_TASK, uc13_pilot = run_uc13_task_signal_pilot()
    uc13.append(("live task-signal pilot", uc13_pilot))
    uc13.append(("live numeric-only active config", run_uc13_live_numeric()))
    uc13.append(("live LLM active config", run_spec_seeds(
        uc13_live_llm_spec(), seeds=DIAGNOSTIC_SEEDS,
        level_id="o1_setup", run_name="mem_uc13_live_llm")))
else:
    uc13.append(("live numeric-only active config", {
        "scores": [], "initial": None, "wall_s": None, "eval_calls": None,
        "artifact": "(offline preflight only: set LIVE=True for real Trace-Bench head-to-head)",
        "artifact_id": None, "artifact_file": None, "spec_file": None,
        "errors": [], "dry": True,
    }))

show_table("Use Case 13 - numeric config optimizer head-to-head", uc13)


/home/xav/miniconda3/envs/humanllm/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Evaluating agent (iteration 0):   0%|                     | 0/4 [00:00<?, ?it/s]

Evaluating agent (iteration 0):  25%|███▎         | 1/4 [00:01<00:04,  1.64s/it]

Evaluating agent (iteration 0):  50%|██████▌      | 2/4 [00:01<00:01,  1.27it/s]

Evaluating agent (iteration 0):  75%|█████████▊   | 3/4 [00:02<00:00,  1.40it/s]

...

Evaluating agent (iteration 0):  83%|██████████▊  | 5/6 [00:03<00:00,  2.16it/s]

Evaluating agent (iteration 0): 100%|█████████████| 6/6 [00:03<00:00,  2.82it/s]

Evaluating agent (iteration 0): 100%|█████████████| 6/6 [00:03<00:00,  1.80it/s]

[Step 0] Average test score: 0.19608700224503586


KeyboardInterrupt: 

In [ ]:
# Final rerun roll-up: includes any use-case variables executed in this kernel.
FINAL_USE_CASES = [
    ("UC1 component code", "uc1"),
    ("UC2 setup/config", "uc2"),
    ("UC3 capability", "uc3"),
    ("UC4 family/transfer", "uc4"),
    ("UC5 optimizer/tool", "uc5"),
    ("UC6 trace feedback", "uc6"),
    ("UC7 graph/suboptimizer", "uc7"),
    ("UC8 campaign policy", "uc8"),
    ("UC9 agentic trace policy", "uc9"),
    ("UC10 promotion policy", "uc10"),
    ("UC11 prompt emitter", "uc11"),
    ("UC12 promoted primitives", "uc12"),
    ("UC13 numeric config", "uc13"),
]

available = [(label, globals()[var]) for label, var in FINAL_USE_CASES if var in globals()]
flat = ["| use case | experiment | initial | mean score | delta | std | n | wall_s | eval/trials | best step | artifact version | best artifact file | spec file | notes | best? |",
        "|---|---|---:|---:|---:|---:|---:|---:|---:|---:|---|---|---|---|---|"]
for uc, data in available:
    best = best_of(data)
    best_label = best[0] if best else None
    for label, result in data:
        scores = _finite(result.get("scores", []))
        mean = statistics.mean(scores) if scores else None
        std = statistics.pstdev(scores) if len(scores) > 1 else None
        delta = (mean - result["initial"]) if mean is not None and result.get("initial") is not None else None
        flat.append(f"| {_md_cell(uc)} | {_md_cell(label)} | {_fmt(result.get('initial'))} | {_fmt(mean)} | {_fmt(delta)} | "
                    f"{_fmt(std)} | {len(scores)} | {_fmt(result.get('wall_s'))} | {_fmt_turn(_result_eval_calls(result))} | {_fmt_turn(result.get('best_step'))} | {_fmt_turn(_artifact_version(result))} | "
                    f"{_md_code(result.get('artifact_file') or '-')} | {_md_code(result.get('spec_file') or '-')} | "
                    f"{_md_cell(_notes_for_result(result))} | {'yes' if label == best_label else ''} |")
_display_markdown("### Final current-kernel rerun table\n" + "\n".join(flat))

best_rows = ["| use case | best experiment | initial | mean score | delta | n | wall_s | best artifact file |",
             "|---|---|---:|---:|---:|---:|---:|---|"]
for uc, data in available:
    b = best_of(data)
    if b is None:
        best_rows.append(f"| {_md_cell(uc)} | (no live result) | - | - | - | 0 | - | - |")
        continue
    label, result = b
    mean = _result_mean(result)
    delta = (mean - result["initial"]) if mean is not None and result.get("initial") is not None else None
    best_rows.append(f"| {_md_cell(uc)} | {_md_cell(label)} | {_fmt(result.get('initial'))} | {_fmt(mean)} | {_fmt(delta)} | "
                     f"{len(_finite(result.get('scores', [])))} | {_fmt(result.get('wall_s'))} | {_md_code(result.get('artifact_file') or '-')} |")
_display_markdown("### Final best final-score table\n" + "\n".join(best_rows))

gain_rows = ["| use case | best positive non-control gain | initial | mean score | delta | n | wall_s | eval/trials | best artifact file |",
             "|---|---|---:|---:|---:|---:|---:|---:|---|"]
for uc, data in available:
    g = best_gain_of(data)
    if g is None:
        gain_rows.append(f"| {_md_cell(uc)} | (no positive non-control gain) | - | - | - | 0 | - | - | - |")
        continue
    label, result = g
    mean = _result_mean(result)
    delta = _result_delta(result)
    gain_rows.append(f"| {_md_cell(uc)} | {_md_cell(label)} | {_fmt(result.get('initial'))} | {_fmt(mean)} | {_fmt(delta)} | "
                     f"{len(_finite(result.get('scores', [])))} | {_fmt(result.get('wall_s'))} | {_fmt_turn(_result_eval_calls(result))} | {_md_code(result.get('artifact_file') or '-')} |")
_display_markdown("### Final best positive non-control gain table\n" + "\n".join(gain_rows))

past = summarize_past_runs(OUTPUT_ROOT.parent)
if past:
    if "_past_runs_table_with_progress" in globals():
        historical = _past_runs_table_with_progress(past)
    else:
        lines = ["| run | use case | initial mean | final mean | best score | n memory dirs | best artifact file |",
                 "|---|---|---:|---:|---:|---:|---|"]
        for row in past:
            lines.append(f"| {_md_cell(row['run'])} | {_md_cell(row['use_case'])} | {_fmt(row['initial_mean'])} | "
                         f"{_fmt(row['final_mean'])} | {_fmt(row['best_score'])} | {row['n_dirs']} | "
                         f"{_md_code(row['artifact_file'])} |")
        historical = "\n".join(lines)
    _display_markdown("### Historical persisted-artifact summary after this run\n" + historical)
else:
    _display_markdown("### Historical persisted-artifact summary after this run\nNo persisted artifacts found.")

_display_markdown("**UC13 interpretation:** the offline row is a deterministic causal preflight for the numeric bridge only. "
                  "The live rows are the benchmark evidence: they use the real Trace-Bench adapter with active `batch_design`/`batch_size`, "
                  "`inner_steps=2`, and the same tight optimizer budget envelope.")


## Consolidated live results after UC13 fixes

Generated from persisted live outputs, not from cached notebook variables. UC2/UC4/UC6 come from `use_cases_uc13_live_fix2_20260618_000000`; UC13 comes from `use_cases_uc13_live_uc13only_20260619_000000`. The `initial/probe` column is an independent initial probe where available; otherwise it is the first persisted starting candidate for that arm.

| UC | experiment | initial/probe | best/final | delta | best artifact/content | output folder |
|---|---|---:|---:|---:|---|---|
| UC2 | QASPER LLM config | 0.1963 | 0.1800 | -0.0163 | `qasper:config:0:63687` starting_artifact: | `examples/notebook_outputs/recursive_opt_use_cases/use_cases_uc13_live_fix2_20260618_000000/mem_uc2_qasper_0` |
| UC2 | QASPER causal numeric config | 0.0172 | 0.1682 | 0.1509 | `qasper_numeric:config:0:57143` batch_design: diversity; batch_size: 6 | `examples/notebook_outputs/recursive_opt_use_cases/use_cases_uc13_live_fix2_20260618_000000/mem_uc2_qasper_numeric_0` |
| UC2 | Mixed GSM8K+QASPER LLM config | -0.0252 | 0.0463 | 0.0716 | `mixed_reasoning:config:0:33598` starting_artifact: | `examples/notebook_outputs/recursive_opt_use_cases/use_cases_uc13_live_fix2_20260618_000000/mem_uc2_mixed_gsm8k_qasper_0` |
| UC2 | DROP LLM config | 0.7500 | 1.0000 | 0.2500 | `drop:config:0:77660` starting_artifact: | `examples/notebook_outputs/recursive_opt_use_cases/use_cases_uc13_live_fix2_20260618_000000/mem_uc2_drop_0` |
| UC4 | O2 family policy | 0.0068 | 0.0291 | 0.0223 | `*:family_policy:0:50185` gsm8k => starting_artifact=; qasper => starting_artifact= | `examples/notebook_outputs/recursive_opt_use_cases/use_cases_uc13_live_fix2_20260618_000000/mem_uc4_o2_policy_0` |
| UC4 | O2 causal numeric family policy | 0.0245 | 0.0245 | 0.0000 | `*:family_policy:0:25835` gsm8k => batch_design=random, batch_size=4; qasper => batch_design=random, batch_size=4 | `examples/notebook_outputs/recursive_opt_use_cases/use_cases_uc13_live_fix2_20260618_000000/mem_uc4_o2_numeric_0` |
| UC4 | O3 cold prior | 0.1342 | 0.1892 | 0.0550 | `*:prior:0:26907` starting_artifact: | `examples/notebook_outputs/recursive_opt_use_cases/use_cases_uc13_live_fix2_20260618_000000/mem_uc4_o3_cold_0` |
| UC4 | O3 warm prior | 0.1922 | 0.2269 | 0.0347 | `*:prior:0:36264` starting_artifact: | `examples/notebook_outputs/recursive_opt_use_cases/use_cases_uc13_live_fix2_20260618_000000/mem_uc4_o3_warm_0` |
| UC6 | trace_type=internal | 0.1567 | 0.2014 | 0.0447 | `reasoning:config:0:88422` starting_artifact: | `examples/notebook_outputs/recursive_opt_use_cases/use_cases_uc13_live_fix2_20260618_000000/mem_uc6_trace_internal_0` |
| UC6 | trace_type=otel | 0.1312 | 0.1877 | 0.0564 | `reasoning:config:0:35554` starting_artifact: | `examples/notebook_outputs/recursive_opt_use_cases/use_cases_uc13_live_fix2_20260618_000000/mem_uc6_trace_otel_0` |
| UC6 | trace_type=hybrid | 0.1235 | 0.1862 | 0.0627 | `reasoning:config:0:80745` starting_artifact: | `examples/notebook_outputs/recursive_opt_use_cases/use_cases_uc13_live_fix2_20260618_000000/mem_uc6_trace_hybrid_0` |
| UC6 | trace_type=internal + causal numeric config | 0.1753 | 0.3102 | 0.1349 | `reasoning:config:0:7339` batch_design: random; batch_size: 4 | `examples/notebook_outputs/recursive_opt_use_cases/use_cases_uc13_live_fix2_20260618_000000/mem_uc6_trace_internal_numeric_0` |
| UC13 | offline numeric causal preflight | 0.3250 | 1.0000 | 0.6750 | `` batch_design=failure_balanced, batch_size=8 | `examples/notebook_outputs/recursive_opt_use_cases/use_cases_uc13_live_uc13only_20260619_000000/mem_uc13_offline_numeric` |
| UC13 | live numeric config search | -0.1655 | -0.1590 | 0.0065 | `` batch_design=random, batch_size=1 | `examples/notebook_outputs/recursive_opt_use_cases/use_cases_uc13_live_uc13only_20260619_000000/mem_uc13_live_numeric` |
| UC13 | live LLM config search | -0.1628 | -0.1608 | 0.0020 | `uc13:config:0:97599` batch_design: random; batch_size: 4 | `examples/notebook_outputs/recursive_opt_use_cases/use_cases_uc13_live_uc13only_20260619_000000/mem_uc13_live_llm_0` |

Key readout: UC13 proves the numeric optimizer path on a deterministic causal preflight (`0.3250 -> 1.0000`, selecting `failure_balanced, batch_size=8`). On the real live benchmark with the tight 8-trial budget, numeric search improves only slightly (`-0.1655 -> -0.1590`) and the LLM config arm lands nearby (`-0.1608`), so the live task is currently low-signal/noisy for this pair of fields. UC6 is the strongest live improvement from the new causal numeric arm (`0.1753 -> 0.3102` using first persisted candidate as the starting comparison).

Several arms still show `final evaluation failed; selected best saved candidate` in artifact metrics. That is now recoverable and correctly persisted, but it points to a remaining trainer/runtime stability issue rather than an optimizer-selection issue.

---
## Three-way benchmark (initial vs standard Trace vs recursive, equal total budget)

The cells above characterize each surface. This section adds the **rigorous benchmark** the
project needs: for a use case, compare three arms at **equal total candidate budget N** —
`initial` (no optimization), `standard` (one-level standard Trace optimization), and
`recursive` (multi-level / prior-carry / specialized sub-optimizer). It reports learning
curves (so recursion can win on **speed**, not only final score) and writes the three artifact
diffs. Helper: `examples/recursive_opt_three_way.py`.

**Fairness:** candidates are split deterministically across levels so standard (1 level) and
recursive (2–3 levels) consume the *same actual total*; the global eval/optimizer/wall caps are
a backstop. **Verdict** credits a recursive win on higher final OR higher best OR fewer
candidates-to-standard-best OR fewer optimizer-LLM-calls; otherwise it emits a diagnostic
bucket (curve-too-short / flat-surface / check-design) instead of a blanket 'badly designed'.

In [12]:
# Three-way benchmark helper (equal-total-budget; learning curves; artifact diffs).
from examples.recursive_opt_three_way import (benchmark_uc, run_numeric_arm, make_code_arm,
                                             markdown_report, bbeh_direct_solver_baseline)

# Equal-budget knobs for the benchmark (total candidates is the fairness unit).
TW_TOTAL_CANDIDATES = int(os.environ.get("RECURSIVE_OPT_TW_CANDIDATES", max(18, RUN_ITERATIONS * NUM_CANDIDATES * 4)))
TW_NUM_CANDIDATES   = NUM_CANDIDATES
# Reliability: use >=3 seeds for every three-way verdict (n=1 is noise; see analysis).
TW_SEEDS            = list(SEEDS) if len(SEEDS) >= 3 else [0, 1, 2]
TW_OPTIMIZER_CALLS  = int(os.environ.get("RECURSIVE_OPT_TW_OPT_CALLS", 12))
TW_EVAL_CALLS       = int(os.environ.get("RECURSIVE_OPT_TW_EVAL_CALLS", 64))
TW_WALL_S           = int(os.environ.get("RECURSIVE_OPT_TW_WALL_S", 1800))
print("three-way budget: total_candidates=%d num_candidates=%d opt_calls=%d eval_calls=%d wall_s=%d"
      % (TW_TOTAL_CANDIDATES, TW_NUM_CANDIDATES, TW_OPTIMIZER_CALLS, TW_EVAL_CALLS, TW_WALL_S))

three-way budget: total_candidates=6 num_candidates=2 opt_calls=4 eval_calls=32 wall_s=900


In [13]:
# --- Three-way: config/prompt UC (standard cold single-level vs recursive warm prior) ---
# initial & standard optimize the prompt on one level; recursive reuses a prior AND adds an
# active field so it is STRUCTURALLY different (this is what a fair recursive arm must be).
_qasper = HARD_PROMPT_TASKS["qasper"]
tw_uc2 = benchmark_uc(
    "UC2_prompt_config_qasper",
    initial   = config_spec(["starting_artifact"], task=_qasper, family_name="reasoning",
                            inner_steps=2, numeric_constraints=CAUSAL_NUMERIC_CONSTRAINTS),
    standard  = {**config_spec(["starting_artifact"], task=_qasper, family_name="reasoning",
                               inner_steps=2, numeric_constraints=CAUSAL_NUMERIC_CONSTRAINTS),
                 "reuse_priors": False},
    recursive = {**config_spec(["starting_artifact", "batch_design", "batch_size"], task=_qasper,
                               family_name="reasoning", inner_steps=2,
                               numeric_constraints=CAUSAL_NUMERIC_CONSTRAINTS),
                 "reuse_priors": True},
    output_root=OUTPUT_ROOT, total_candidates=TW_TOTAL_CANDIDATES, num_candidates=TW_NUM_CANDIDATES,
    optimizer_llm_calls=TW_OPTIMIZER_CALLS, eval_llm_calls=TW_EVAL_CALLS, wall_time_s=TW_WALL_S,
    seeds=TW_SEEDS, primary_level="o1_setup",
    notes="recursive = warm prior + active numeric fields vs standard cold prompt-only") if LIVE else None
display(Markdown(markdown_report(tw_uc2))) if tw_uc2 else print("set LIVE=True to run the three-way UC2 benchmark")

/home/xav/miniconda3/envs/humanllm/lib/python3.12/site-packages/torch/cuda/__init__.py:180: UserWarning: CUDA initialization: The NVIDIA driver on your system is too old (found version 12020). Please update your GPU driver by downloading and installing a new version from the URL: http://www.nvidia.com/Download/index.aspx Alternatively, go to: https://pytorch.org to install a PyTorch version that has been compiled with your version of the CUDA driver. (Triggered internally at /pytorch/c10/cuda/CUDAFunctions.cpp:119.)
  return torch._C._cuda_getDeviceCount() > 0
/home/xav/miniconda3/envs/humanllm/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


PrioritySearch initialized with only long-term memory.
Epoch: 0. Iteration: 0


Sampling training minibatch: Sampling 1 agents on 4 inputs:   0%| | 0/4 [00:00<?

Sampling training minibatch: Sampling 1 agents on 4 inputs:  25%|▎| 1/4 [00:01<0

Sampling training minibatch: Sampling 1 agents on 4 inputs:  50%|▌| 2/4 [00:01<0

...

Backward:   0%|                                           | 0/1 [00:00<?, ?it/s]

Backward: 100%|█████████████████████████████████| 1/1 [00:00<00:00, 2404.99it/s]

Calling optimizers: Generating 1 proposals for each of 1 batches:   0%| | 0/1 [0

Calling optimizers: Generating 1 proposals for each of 1 batches:   0%| | 0/1 [0

### UC2_prompt_config_qasper

| arm | mean final | mean best | n | err | best@unit |
|---|---:|---:|---:|---:|---:|
| initial | 0.213 | 0.213 | 3 | 0 | 0.000 |
| standard | 0.261 | 0.261 | 3 | 0 | 6.000 |
| recursive | 0.280 | 0.280 | 3 | 0 | 6.000 |

- **verdict:** `recursive_wins_final` — recursive final 0.280 > standard 0.261
- recursive − standard (final): `0.019`  |  (best): `0.019`
- speed: recursive reaches standard best @ candidate `6` (standard best @ `6`)
- diffs in `uc2_prompt_config_qasper/diffs/` (initial→standard, initial→recursive, standard→recursive)
- notes: recursive = warm prior + active numeric fields vs standard cold prompt-only

In [14]:
# --- Three-way: UC13 numeric config optimizer head-to-head ---
# standard = generative optimization over numeric/categorical fields; recursive = Optuna route
# (optimize_config_numeric) at the SAME candidate budget. The expected win is SPEED/COST.
_uc13_base = config_spec(["batch_design", "batch_size"], task=FAMILY_TASK, family_name="reasoning",
                         inner_steps=2, numeric_constraints=CAUSAL_NUMERIC_CONSTRAINTS)
_uc13_recursive = {**_uc13_base, "numeric": {
    "level_id": "o1_setup", "task": FAMILY_TASK,
    "fields": ["batch_design", "batch_size"], "optimizer": "optuna",
    "space": {"batch_design": ("cat", ("random","failure_balanced","curriculum","diversity")),
              "batch_size": ("cat", (2,4,8))}}}
tw_uc13 = benchmark_uc(
    "UC13_numeric_head_to_head",
    initial   = _uc13_base,
    standard  = {**_uc13_base, "reuse_priors": False},
    recursive = _uc13_recursive,
    output_root=OUTPUT_ROOT, total_candidates=TW_TOTAL_CANDIDATES, num_candidates=TW_NUM_CANDIDATES,
    optimizer_llm_calls=TW_OPTIMIZER_CALLS, eval_llm_calls=TW_EVAL_CALLS, wall_time_s=TW_WALL_S,
    seeds=SEEDS, primary_level="o1_setup", recursive_runner=run_numeric_arm,
    notes="recursive/meta numeric optimizer vs generative; win = faster/cheaper to standard's best") if LIVE else None
display(Markdown(markdown_report(tw_uc13))) if tw_uc13 else print("set LIVE=True to run the three-way UC13 benchmark")

PrioritySearch initialized with only long-term memory.
Epoch: 0. Iteration: 0


Sampling training minibatch: Sampling 1 agents on 4 inputs:   0%| | 0/4 [00:00<?

Sampling training minibatch: Sampling 1 agents on 4 inputs:  25%|▎| 1/4 [00:01<0

Sampling training minibatch: Sampling 1 agents on 4 inputs:  75%|▊| 3/4 [00:01<0

Sampling training minibatch: Sampling 1 agents on 4 inputs: 100%|█| 4/4 [00:01<0

...

Evaluating agent (iteration 0):  75%|█████████▊   | 6/8 [00:02<00:00,  2.75it/s]

Evaluating agent (iteration 0):  88%|███████████▍ | 7/8 [00:02<00:00,  2.76it/s]

Evaluating agent (iteration 0): 100%|█████████████| 8/8 [00:02<00:00,  2.79it/s]

[Step 0] Average test score: 0.0


### UC13_numeric_head_to_head

| arm | mean final | mean best | n | err | best@unit |
|---|---:|---:|---:|---:|---:|
| initial | -0.161 | -0.161 | 3 | 0 | 0.000 |
| standard | -0.160 | -0.160 | 3 | 0 | 6.000 |
| recursive | -0.158 | -0.158 | 3 | 0 | 1.333 |

- **verdict:** `recursive_wins_final` — recursive final -0.158 > standard -0.160
- recursive − standard (final): `0.002`  |  (best): `0.002`
- speed: recursive reaches standard best @ candidate `2` (standard best @ `6`)
- diffs in `uc13_numeric_head_to_head/diffs/` (initial→standard, initial→recursive, standard→recursive)
- notes: recursive/meta numeric optimizer vs generative; win = faster/cheaper to standard's best

In [15]:
# --- Three-way: UC4 family-policy / prior transfer (standard cold vs recursive warm O2->O3) ---
# This surface already has the warm/cold switch. standard = cold single O2 policy; recursive =
# warm O2->O3 (carries a transferable prior) at the SAME total candidate budget.
tw_uc4 = benchmark_uc(
    "UC4_family_policy_prior",
    initial   = family_policy_spec("o2", warm=False, targets=CAUSAL_NUMERIC_TARGETS,
                                   inner_steps=2, constraints=CAUSAL_NUMERIC_CONSTRAINTS),
    standard  = {**family_policy_spec("o2", warm=False, targets=CAUSAL_NUMERIC_TARGETS,
                                      inner_steps=2, constraints=CAUSAL_NUMERIC_CONSTRAINTS),
                 "reuse_priors": False},
    recursive = {**family_policy_spec("o3", warm=True, targets=CAUSAL_NUMERIC_TARGETS,
                                      inner_steps=2, constraints=CAUSAL_NUMERIC_CONSTRAINTS),
                 "reuse_priors": True},
    output_root=OUTPUT_ROOT, total_candidates=TW_TOTAL_CANDIDATES, num_candidates=TW_NUM_CANDIDATES,
    optimizer_llm_calls=TW_OPTIMIZER_CALLS, eval_llm_calls=TW_EVAL_CALLS, wall_time_s=TW_WALL_S,
    seeds=TW_SEEDS, primary_level={"initial": "o2_policy", "standard": "o2_policy", "recursive": "o3_prior"},
    notes="recursive = warm O2->O3 prior transfer vs standard cold single-level policy") if LIVE else None
display(Markdown(markdown_report(tw_uc4))) if tw_uc4 else print("set LIVE=True to run the three-way UC4 benchmark")

Evaluating agent (iteration 0):   0%|                     | 0/4 [00:00<?, ?it/s]

Evaluating agent (iteration 0):  25%|███▎         | 1/4 [00:01<00:03,  1.09s/it]

Evaluating agent (iteration 0):  75%|█████████▊   | 3/4 [00:01<00:00,  2.50it/s]

Evaluating agent (iteration 0): 100%|█████████████| 4/4 [00:01<00:00,  2.79it/s]

[Step 0] Average test score: 0.0


...

Evaluating agent (iteration 0):  75%|█████████▊   | 3/4 [00:02<00:00,  1.69it/s]

Evaluating agent (iteration 0): 100%|█████████████| 4/4 [00:02<00:00,  1.56it/s]

Evaluating agent (iteration 0): 100%|█████████████| 4/4 [00:02<00:00,  1.36it/s]

[Step 0] Average test score: 0.1673706371807726


### UC4_family_policy_prior

| arm | mean final | mean best | n | err | best@unit |
|---|---:|---:|---:|---:|---:|
| initial | 0.004 | 0.004 | 3 | 0 | 0.000 |
| standard | 0.029 | 0.029 | 3 | 0 | 6.000 |
| recursive | 0.192 | 0.192 | 3 | 0 | 6.000 |

- **verdict:** `recursive_wins_final` — recursive final 0.192 > standard 0.029
- recursive − standard (final): `0.163`  |  (best): `0.163`
- speed: recursive reaches standard best @ candidate `6` (standard best @ `6`)
- diffs in `uc4_family_policy_prior/diffs/` (initial→standard, initial→recursive, standard→recursive)
- notes: recursive = warm O2->O3 prior transfer vs standard cold single-level policy

In [16]:
# --- Three-way: UC1 code surface (standard cold rewrite vs recursive warm two-phase prior) ---
# Code UCs use make_code_arm. standard = cold ComponentSpec optimized for the full budget;
# recursive = two-phase at the SAME total budget (phase-1 promotes a prior, phase-2 warm-starts
# from it). All three arms share one baseline + evaluator so the diff is meaningful.
from opto.features.recursive_opt.tracebench import make_tracebench_direct_answer_evaluator
_uc1_task = "internal:multiobjective_bbeh"
_uc1_eval = make_tracebench_direct_answer_evaluator(_uc1_task, max_examples=MAX_EXAMPLES, normalizer=_norm_bool_answer)
_uc1_base = _BASELINES["bbeh_direct_solver"]
_uc1_obj  = "Rewrite the BBEH boolean-expression solver to return the correct True/False."
_uc1_arm_kwargs = dict(baseline=_uc1_base, evaluate=_uc1_eval, task_id=_uc1_task, objective=_uc1_obj)
_uc1_spec = {"_component": "bbeh_direct_solver", "_max_examples": MAX_EXAMPLES}
tw_uc1 = benchmark_uc(
    "UC1_code_bbeh_solver",
    initial   = _uc1_spec, standard = _uc1_spec, recursive = _uc1_spec,
    output_root=OUTPUT_ROOT, total_candidates=TW_TOTAL_CANDIDATES, num_candidates=TW_NUM_CANDIDATES,
    optimizer_llm_calls=TW_OPTIMIZER_CALLS, eval_llm_calls=TW_EVAL_CALLS, wall_time_s=TW_WALL_S,
    seeds=TW_SEEDS,
    initial_runner   = make_code_arm(warm=False, **_uc1_arm_kwargs),
    standard_runner  = make_code_arm(warm=False, **_uc1_arm_kwargs),
    recursive_runner = make_code_arm(warm=True,  **_uc1_arm_kwargs),
    notes="recursive = two-phase warm prior (phase1 promote -> phase2 warm-start) vs cold rewrite") if LIVE else None
display(Markdown(markdown_report(tw_uc1))) if tw_uc1 else print("set LIVE=True to run the three-way UC1 benchmark")

PrioritySearch initialized with only long-term memory.
Epoch: 0. Iteration: 0


Sampling training minibatch: Sampling 2 agents on 1 inputs:   0%| | 0/2 [00:00<?

Sampling training minibatch: Sampling 2 agents on 1 inputs: 100%|█| 2/2 [00:00<0

Evaluating agent:   0%|                                   | 0/8 [00:00<?, ?it/s]

Evaluating agent: 100%|█████████████████████████| 8/8 [00:00<00:00, 9005.48it/s]

...

Sampling training minibatch: Sampling 2 agents on 1 inputs: 100%|█| 2/2 [00:00<0

Evaluating agent:   0%|                                   | 0/8 [00:00<?, ?it/s]

Evaluating agent: 100%|█████████████████████████| 8/8 [00:00<00:00, 2484.04it/s]

[Step 1] Test/test_score: 1.0
[Step 1] Algo/Average train score: 0.71875
[Step 1] Update/n_iters: 1
[Step 1] Update/short_term_memory_size: 0
[Step 1] Update/long_term_memory_size: 3
[Step 1] Update/using_short_term_memory: False
[Step 1] Update/using_long_term_memory: True
[Step 1] Update/total_samples: 4
[Step 1] Update/best_candidate_priority: 1.0
[Step 1] Update/best_candidate_mean_score: 1.0
[Step 1] Update/best_candidate_num_rollouts: 1
[Step 1] Update/num_exploration_candidates: 2
[Step 1] Update/exploration_candidates_mean_priority: 0.8125
[Step 1] Update/exploration_candidates_mean_score: 0.8125
[Step 1] Update/exploration_candidates_average_num_rollouts: 1.5
[Step 1] Sample/mean_score: 0.8125
[Step 1] Sample/num_samples: 2
[Step 1] Sample/self.n_epochs: 0
[Step 1] Algo/Number of training samples: 4
[Step 1] Parameter/__code:9: def _bbeh_direct_solver(self, question):
    """Return True/False for a BBEH boolean expression ending with ' is'."""
    import re

    # Expect forma

### UC1_code_bbeh_solver

| arm | mean final | mean best | n | err | best@unit |
|---|---:|---:|---:|---:|---:|
| initial | 0.625 | 0.625 | 3 | 0 | 0.000 |
| standard | 1.000 | 1.000 | 3 | 0 | 6.000 |
| recursive | 1.000 | 1.000 | 3 | 0 | 6.000 |

- **verdict:** `recursive_wins_optimizer_calls` — recursive matched standard best 1.000 with fewer optimizer LLM calls (2.0 < 4.0)
- recursive − standard (final): `0.000`  |  (best): `0.000`
- speed: recursive reaches standard best @ candidate `6` (standard best @ `6`)
- diffs in `uc1_code_bbeh_solver/diffs/` (initial→standard, initial→recursive, standard→recursive)
- notes: recursive = two-phase warm prior (phase1 promote -> phase2 warm-start) vs cold rewrite

In [17]:
# --- Three-way: UC9 agentic tool+hint policy (code surface; standard cold vs recursive warm) ---
# UC9 has the largest single-arm gain (+0.635) and ample headroom -> strong three-way candidate.
# Uses the real agentic baseline + evaluator. standard = cold policy rewrite; recursive =
# two-phase warm prior via make_code_arm(warm=True).
_uc9_task = "internal:agentic_trace_policy"
_uc9_kwargs = dict(baseline=_baseline_agentic_trace_policy,
                   evaluate=evaluate_agentic_trace_policy,
                   task_id=_uc9_task,
                   objective="Rewrite the agentic tool+hint policy to maximise task score.")
_uc9_spec = {"_component": "agentic_trace_policy", "_max_examples": MAX_EXAMPLES}
tw_uc9 = benchmark_uc(
    "UC9_agentic_policy_code",
    initial=_uc9_spec, standard=_uc9_spec, recursive=_uc9_spec,
    output_root=OUTPUT_ROOT, total_candidates=TW_TOTAL_CANDIDATES, num_candidates=TW_NUM_CANDIDATES,
    optimizer_llm_calls=TW_OPTIMIZER_CALLS, eval_llm_calls=TW_EVAL_CALLS, wall_time_s=TW_WALL_S,
    seeds=TW_SEEDS,
    initial_runner=make_code_arm(warm=False, **_uc9_kwargs),
    standard_runner=make_code_arm(warm=False, **_uc9_kwargs),
    recursive_runner=make_code_arm(warm=True, **_uc9_kwargs),
    notes="recursive = two-phase warm prior vs cold rewrite on the agentic policy artifact") if LIVE else None
display(Markdown(markdown_report(tw_uc9))) if tw_uc9 else print("set LIVE=True to run the three-way UC9 benchmark")

PrioritySearch initialized with only long-term memory.
Epoch: 0. Iteration: 0


Sampling training minibatch: Sampling 2 agents on 1 inputs:   0%| | 0/2 [00:00<?

Sampling training minibatch: Sampling 2 agents on 1 inputs: 100%|█| 2/2 [00:00<0

Evaluating agent:   0%|                                   | 0/8 [00:00<?, ?it/s]

Evaluating agent: 100%|█████████████████████████| 8/8 [00:00<00:00, 3335.10it/s]

...

Sampling training minibatch: Sampling 2 agents on 1 inputs: 100%|█| 2/2 [00:00<0

Evaluating agent:   0%|                                   | 0/8 [00:00<?, ?it/s]

Evaluating agent: 100%|█████████████████████████| 8/8 [00:00<00:00, 2273.18it/s]

[Step 1] Test/test_score: 0.86
[Step 1] Algo/Average train score: 0.4475
[Step 1] Update/n_iters: 1
[Step 1] Update/short_term_memory_size: 0
[Step 1] Update/long_term_memory_size: 3
[Step 1] Update/using_short_term_memory: False
[Step 1] Update/using_long_term_memory: True
[Step 1] Update/total_samples: 4
[Step 1] Update/best_candidate_priority: 0.86
[Step 1] Update/best_candidate_mean_score: 0.86
[Step 1] Update/best_candidate_num_rollouts: 1
[Step 1] Update/num_exploration_candidates: 2
[Step 1] Update/exploration_candidates_mean_priority: 0.685
[Step 1] Update/exploration_candidates_mean_score: 0.685
[Step 1] Update/exploration_candidates_average_num_rollouts: 1.0
[Step 1] Sample/mean_score: 0.685
[Step 1] Sample/num_samples: 2
[Step 1] Sample/self.n_epochs: 0
[Step 1] Algo/Number of training samples: 4
[Step 1] Parameter/__code:18: def _baseline_agentic_trace_policy(self, signal):
    s = (signal or "").lower()

    # Avoid expensive tool calls when the system is saturated.
    if

### UC9_agentic_policy_code

| arm | mean final | mean best | n | err | best@unit |
|---|---:|---:|---:|---:|---:|
| initial | 0.210 | 0.210 | 3 | 0 | 0.000 |
| standard | 0.933 | 0.933 | 3 | 0 | 6.000 |
| recursive | 0.840 | 0.840 | 3 | 0 | 6.000 |

- **verdict:** `no_win_check_design` — recursive did not beat standard; check: is the recursive arm structurally different (prior/extra level/numeric route)? budget starved? prior hurt? field inactive?
- recursive − standard (final): `-0.093`  |  (best): `-0.093`
- speed: recursive reaches standard best @ candidate `None` (standard best @ `6`)
- diffs in `uc9_agentic_policy_code/diffs/` (initial→standard, initial→recursive, standard→recursive)
- notes: recursive = two-phase warm prior vs cold rewrite on the agentic policy artifact

In [18]:
# --- Three-way: UC5 optimizer tool policy (code surface; standard cold vs recursive warm two-phase) ---
_optimizer_tool_policy_kwargs = dict(baseline=_BASELINES["optimizer_tool_policy"],
                   evaluate=evaluate_optimizer_tool_policy,
                   task_id="internal:optimizer_tool_policy",
                   objective="Rewrite the optimizer-side tool-selection policy to maximise score.")
_optimizer_tool_policy_spec = {"_component": "optimizer_tool_policy", "_max_examples": MAX_EXAMPLES}
tw_optimizer_tool_policy = benchmark_uc(
    "UC5_tool_policy_code",
    initial=_optimizer_tool_policy_spec, standard=_optimizer_tool_policy_spec, recursive=_optimizer_tool_policy_spec,
    output_root=OUTPUT_ROOT, total_candidates=TW_TOTAL_CANDIDATES, num_candidates=TW_NUM_CANDIDATES,
    optimizer_llm_calls=TW_OPTIMIZER_CALLS, eval_llm_calls=TW_EVAL_CALLS, wall_time_s=TW_WALL_S,
    seeds=TW_SEEDS,
    initial_runner=make_code_arm(warm=False, **_optimizer_tool_policy_kwargs),
    standard_runner=make_code_arm(warm=False, **_optimizer_tool_policy_kwargs),
    recursive_runner=make_code_arm(warm=True, **_optimizer_tool_policy_kwargs),
    notes="recursive = two-phase warm prior vs cold rewrite") if LIVE else None
display(Markdown(markdown_report(tw_optimizer_tool_policy))) if tw_optimizer_tool_policy else print("set LIVE=True to run UC5_tool_policy_code")

PrioritySearch initialized with only long-term memory.
Epoch: 0. Iteration: 0


Sampling training minibatch: Sampling 2 agents on 1 inputs:   0%| | 0/2 [00:00<?

Sampling training minibatch: Sampling 2 agents on 1 inputs: 100%|█| 2/2 [00:00<0

Evaluating agent:   0%|                                   | 0/8 [00:00<?, ?it/s]

Evaluating agent: 100%|█████████████████████████| 8/8 [00:00<00:00, 4866.49it/s]

...

Sampling training minibatch: Sampling 2 agents on 1 inputs: 100%|█| 2/2 [00:00<0

Evaluating agent:   0%|                                   | 0/8 [00:00<?, ?it/s]

Evaluating agent: 100%|████████████████████████| 8/8 [00:00<00:00, 20932.27it/s]

[Step 1] Test/test_score: 1.0
[Step 1] Algo/Average train score: 0.6875
[Step 1] Update/n_iters: 1
[Step 1] Update/short_term_memory_size: 0
[Step 1] Update/long_term_memory_size: 3
[Step 1] Update/using_short_term_memory: False
[Step 1] Update/using_long_term_memory: True
[Step 1] Update/total_samples: 4
[Step 1] Update/best_candidate_priority: 1.0
[Step 1] Update/best_candidate_mean_score: 1.0
[Step 1] Update/best_candidate_num_rollouts: 1
[Step 1] Update/num_exploration_candidates: 2
[Step 1] Update/exploration_candidates_mean_priority: 1.0
[Step 1] Update/exploration_candidates_mean_score: 1.0
[Step 1] Update/exploration_candidates_average_num_rollouts: 1.0
[Step 1] Sample/mean_score: 1.0
[Step 1] Sample/num_samples: 2
[Step 1] Sample/self.n_epochs: 0
[Step 1] Algo/Number of training samples: 4
[Step 1] Parameter/__code:27: def _baseline_tool_policy(self, signal):
    s = (signal or "").lower()
    tools = ["note"]
    if ("prior failures" in s) or ("family examples" in s):
       

### UC5_tool_policy_code

| arm | mean final | mean best | n | err | best@unit |
|---|---:|---:|---:|---:|---:|
| initial | 0.375 | 0.375 | 3 | 0 | 0.000 |
| standard | 1.000 | 1.000 | 3 | 0 | 6.000 |
| recursive | 0.917 | 0.917 | 3 | 0 | 6.000 |

- **verdict:** `no_win_check_design` — recursive did not beat standard; check: is the recursive arm structurally different (prior/extra level/numeric route)? budget starved? prior hurt? field inactive?
- recursive − standard (final): `-0.083`  |  (best): `-0.083`
- speed: recursive reaches standard best @ candidate `None` (standard best @ `6`)
- diffs in `uc5_tool_policy_code/diffs/` (initial→standard, initial→recursive, standard→recursive)
- notes: recursive = two-phase warm prior vs cold rewrite

In [19]:
# --- Three-way: UC8 meta-campaign policy (code surface; standard cold vs recursive warm two-phase) ---
_campaign_policy_kwargs = dict(baseline=_BASELINES["campaign_policy"],
                   evaluate=evaluate_campaign_policy,
                   task_id="internal:campaign_policy",
                   objective="Rewrite the meta-campaign controller to maximise score.")
_campaign_policy_spec = {"_component": "campaign_policy", "_max_examples": MAX_EXAMPLES}
tw_campaign_policy = benchmark_uc(
    "UC8_campaign_policy_code",
    initial=_campaign_policy_spec, standard=_campaign_policy_spec, recursive=_campaign_policy_spec,
    output_root=OUTPUT_ROOT, total_candidates=TW_TOTAL_CANDIDATES, num_candidates=TW_NUM_CANDIDATES,
    optimizer_llm_calls=TW_OPTIMIZER_CALLS, eval_llm_calls=TW_EVAL_CALLS, wall_time_s=TW_WALL_S,
    seeds=TW_SEEDS,
    initial_runner=make_code_arm(warm=False, **_campaign_policy_kwargs),
    standard_runner=make_code_arm(warm=False, **_campaign_policy_kwargs),
    recursive_runner=make_code_arm(warm=True, **_campaign_policy_kwargs),
    notes="recursive = two-phase warm prior vs cold rewrite") if LIVE else None
display(Markdown(markdown_report(tw_campaign_policy))) if tw_campaign_policy else print("set LIVE=True to run UC8_campaign_policy_code")

PrioritySearch initialized with only long-term memory.
Epoch: 0. Iteration: 0


Sampling training minibatch: Sampling 2 agents on 1 inputs:   0%| | 0/2 [00:00<?

Sampling training minibatch: Sampling 2 agents on 1 inputs: 100%|█| 2/2 [00:00<0

Evaluating agent:   0%|                                   | 0/8 [00:00<?, ?it/s]

Evaluating agent: 100%|████████████████████████| 8/8 [00:00<00:00, 10764.98it/s]

...

Sampling training minibatch: Sampling 2 agents on 1 inputs: 100%|█| 2/2 [00:00<0

Evaluating agent:   0%|                                   | 0/8 [00:00<?, ?it/s]

Evaluating agent: 100%|█████████████████████████| 8/8 [00:00<00:00, 5541.61it/s]

[Step 1] Test/test_score: 0.515
[Step 1] Algo/Average train score: 0.4225
[Step 1] Update/n_iters: 1
[Step 1] Update/short_term_memory_size: 0
[Step 1] Update/long_term_memory_size: 3
[Step 1] Update/using_short_term_memory: False
[Step 1] Update/using_long_term_memory: True
[Step 1] Update/total_samples: 4
[Step 1] Update/best_candidate_priority: 0.515
[Step 1] Update/best_candidate_mean_score: 0.515
[Step 1] Update/best_candidate_num_rollouts: 1
[Step 1] Update/num_exploration_candidates: 2
[Step 1] Update/exploration_candidates_mean_priority: 0.485
[Step 1] Update/exploration_candidates_mean_score: 0.485
[Step 1] Update/exploration_candidates_average_num_rollouts: 1.0
[Step 1] Sample/mean_score: 0.485
[Step 1] Sample/num_samples: 2
[Step 1] Sample/self.n_epochs: 0
[Step 1] Algo/Number of training samples: 4
[Step 1] Parameter/__code:36: def _baseline_campaign_policy(self, diagnostics):
    """Return action/task/reason from diagnostics; this seed is intentionally weak."""
    # Diagn

### UC8_campaign_policy_code

| arm | mean final | mean best | n | err | best@unit |
|---|---:|---:|---:|---:|---:|
| initial | 0.360 | 0.360 | 3 | 0 | 0.000 |
| standard | 0.498 | 0.498 | 3 | 0 | 6.000 |
| recursive | 0.478 | 0.478 | 3 | 0 | 6.000 |

- **verdict:** `no_win_check_design` — recursive did not beat standard; check: is the recursive arm structurally different (prior/extra level/numeric route)? budget starved? prior hurt? field inactive?
- recursive − standard (final): `-0.020`  |  (best): `-0.020`
- speed: recursive reaches standard best @ candidate `None` (standard best @ `6`)
- diffs in `uc8_campaign_policy_code/diffs/` (initial→standard, initial→recursive, standard→recursive)
- notes: recursive = two-phase warm prior vs cold rewrite

In [20]:
# --- Three-way: UC10 artifact promotion policy (code surface; standard cold vs recursive warm two-phase) ---
_promotion_policy_kwargs = dict(baseline=_BASELINES["promotion_policy"],
                   evaluate=evaluate_promotion_policy,
                   task_id="internal:artifact_promotion_policy",
                   objective=("Rewrite the artifact-promotion gate to maximise guarded score. "
                             "Return action/reason/confidence. Apply guards in order: "
                             "invalid syntax -> reject/repair; saturated control -> control/archive; "
                             "single seed or weak evidence -> retest/hold; regression -> rollback/cold; "
                             "promote only validated gains."))
_promotion_policy_spec = {"_component": "promotion_policy", "_max_examples": MAX_EXAMPLES}
tw_promotion_policy = benchmark_uc(
    "UC10_promotion_policy_code",
    initial=_promotion_policy_spec, standard=_promotion_policy_spec, recursive=_promotion_policy_spec,
    output_root=OUTPUT_ROOT, total_candidates=TW_TOTAL_CANDIDATES, num_candidates=TW_NUM_CANDIDATES,
    optimizer_llm_calls=TW_OPTIMIZER_CALLS, eval_llm_calls=TW_EVAL_CALLS, wall_time_s=TW_WALL_S,
    seeds=TW_SEEDS,
    initial_runner=make_code_arm(warm=False, **_promotion_policy_kwargs),
    standard_runner=make_code_arm(warm=False, **_promotion_policy_kwargs),
    recursive_runner=make_code_arm(warm=True, **_promotion_policy_kwargs),
    notes="recursive = two-phase warm prior vs cold rewrite") if LIVE else None
display(Markdown(markdown_report(tw_promotion_policy))) if tw_promotion_policy else print("set LIVE=True to run UC10_promotion_policy_code")

PrioritySearch initialized with only long-term memory.
Epoch: 0. Iteration: 0


Sampling training minibatch: Sampling 2 agents on 1 inputs:   0%| | 0/2 [00:00<?

Sampling training minibatch: Sampling 2 agents on 1 inputs: 100%|█| 2/2 [00:00<0

Evaluating agent:   0%|                                   | 0/8 [00:00<?, ?it/s]

Evaluating agent: 100%|█████████████████████████| 8/8 [00:00<00:00, 3804.79it/s]

...

Sampling training minibatch: Sampling 2 agents on 1 inputs: 100%|█| 2/2 [00:00<0

Evaluating agent:   0%|                                   | 0/8 [00:00<?, ?it/s]

Evaluating agent: 100%|█████████████████████████| 8/8 [00:00<00:00, 7553.90it/s]

[Step 1] Test/test_score: 0.5549999999999999
[Step 1] Algo/Average train score: 0.245
[Step 1] Update/n_iters: 1
[Step 1] Update/short_term_memory_size: 0
[Step 1] Update/long_term_memory_size: 3
[Step 1] Update/using_short_term_memory: False
[Step 1] Update/using_long_term_memory: True
[Step 1] Update/total_samples: 4
[Step 1] Update/best_candidate_priority: 0.5549999999999999
[Step 1] Update/best_candidate_mean_score: 0.5549999999999999
[Step 1] Update/best_candidate_num_rollouts: 1
[Step 1] Update/num_exploration_candidates: 2
[Step 1] Update/exploration_candidates_mean_priority: 0.36
[Step 1] Update/exploration_candidates_mean_score: 0.36
[Step 1] Update/exploration_candidates_average_num_rollouts: 1.0
[Step 1] Sample/mean_score: 0.36
[Step 1] Sample/num_samples: 2
[Step 1] Sample/self.n_epochs: 0
[Step 1] Algo/Number of training samples: 4
[Step 1] Parameter/__code:45: def _baseline_promotion_policy(self, artifact_report):
    # artifact_report fields seen in inputs:
    # syntax_

### UC10_promotion_policy_code

| arm | mean final | mean best | n | err | best@unit |
|---|---:|---:|---:|---:|---:|
| initial | 0.130 | 0.130 | 3 | 0 | 0.000 |
| standard | 0.300 | 0.300 | 3 | 0 | 4.000 |
| recursive | 0.285 | 0.285 | 3 | 0 | 4.000 |

- **verdict:** `no_win_check_design` — recursive did not beat standard; check: is the recursive arm structurally different (prior/extra level/numeric route)? budget starved? prior hurt? field inactive?
- recursive − standard (final): `-0.015`  |  (best): `-0.015`
- speed: recursive reaches standard best @ candidate `None` (standard best @ `6`)
- diffs in `uc10_promotion_policy_code/diffs/` (initial→standard, initial→recursive, standard→recursive)
- notes: recursive = two-phase warm prior vs cold rewrite

In [21]:
# --- Three-way: UC11 prompt-emitter artifact (code surface; standard cold vs recursive warm two-phase) ---
_qasper_prompt_emitter_kwargs = dict(baseline=_BASELINES["qasper_prompt_emitter"],
                   evaluate=make_artifact_emitter_evaluator("hf:qasper", max_examples=HARD_MAX_EXAMPLES),
                   task_id="hf:qasper",
                   objective="Rewrite the prompt-emitter code artifact to maximise score.")
_qasper_prompt_emitter_spec = {"_component": "qasper_prompt_emitter", "_max_examples": MAX_EXAMPLES}
tw_qasper_prompt_emitter = benchmark_uc(
    "UC11_prompt_emitter_code",
    initial=_qasper_prompt_emitter_spec, standard=_qasper_prompt_emitter_spec, recursive=_qasper_prompt_emitter_spec,
    output_root=OUTPUT_ROOT, total_candidates=TW_TOTAL_CANDIDATES, num_candidates=TW_NUM_CANDIDATES,
    optimizer_llm_calls=TW_OPTIMIZER_CALLS, eval_llm_calls=TW_EVAL_CALLS, wall_time_s=TW_WALL_S,
    seeds=TW_SEEDS,
    initial_runner=make_code_arm(warm=False, **_qasper_prompt_emitter_kwargs),
    standard_runner=make_code_arm(warm=False, **_qasper_prompt_emitter_kwargs),
    recursive_runner=make_code_arm(warm=True, **_qasper_prompt_emitter_kwargs),
    notes="recursive = two-phase warm prior vs cold rewrite") if LIVE else None
display(Markdown(markdown_report(tw_qasper_prompt_emitter))) if tw_qasper_prompt_emitter else print("set LIVE=True to run UC11_prompt_emitter_code")

PrioritySearch initialized with only long-term memory.
Epoch: 0. Iteration: 0


Sampling training minibatch: Sampling 2 agents on 1 inputs:   0%| | 0/2 [00:00<?

Sampling training minibatch: Sampling 2 agents on 1 inputs:  50%|▌| 1/2 [00:06<0

Sampling training minibatch: Sampling 2 agents on 1 inputs: 100%|█| 2/2 [00:08<0

Sampling training minibatch: Sampling 2 agents on 1 inputs: 100%|█| 2/2 [00:08<0

...

Evaluating agent:  75%|████████████████████▎      | 6/8 [00:21<00:04,  2.07s/it]

Evaluating agent:  88%|███████████████████████▋   | 7/8 [00:22<00:01,  1.73s/it]

Evaluating agent: 100%|███████████████████████████| 8/8 [00:22<00:00,  2.81s/it]

[Step 1] Test/test_score: 0.26088727882266366
[Step 1] Algo/Average train score: 0.16757398728688494
[Step 1] Update/n_iters: 1
[Step 1] Update/short_term_memory_size: 0
[Step 1] Update/long_term_memory_size: 3
[Step 1] Update/using_short_term_memory: False
[Step 1] Update/using_long_term_memory: True
[Step 1] Update/total_samples: 4
[Step 1] Update/best_candidate_priority: 0.1878538647965761
[Step 1] Update/best_candidate_mean_score: 0.1878538647965761
[Step 1] Update/best_candidate_num_rollouts: 1
[Step 1] Update/num_exploration_candidates: 2
[Step 1] Update/exploration_candidates_mean_priority: 0.17313938139726337
[Step 1] Update/exploration_candidates_mean_score: 0.17313938139726337
[Step 1] Update/exploration_candidates_average_num_rollouts: 1.5
[Step 1] Sample/mean_score: 0.17672307657581926
[Step 1] Sample/num_samples: 2
[Step 1] Sample/self.n_epochs: 0
[Step 1] Algo/Number of training samples: 4
[Step 1] Parameter/__code:54: def _qasper_prompt_emitter(self):
    """Weak seed: e

### UC11_prompt_emitter_code

| arm | mean final | mean best | n | err | best@unit |
|---|---:|---:|---:|---:|---:|
| initial | 0.137 | 0.137 | 3 | 0 | 0.000 |
| standard | 0.317 | 0.317 | 3 | 0 | 4.000 |
| recursive | 0.436 | 0.436 | 3 | 0 | 2.000 |

- **verdict:** `recursive_wins_final` — recursive final 0.436 > standard 0.317
- recursive − standard (final): `0.119`  |  (best): `0.119`
- speed: recursive reaches standard best @ candidate `None` (standard best @ `6`)
- diffs in `uc11_prompt_emitter_code/diffs/` (initial→standard, initial→recursive, standard→recursive)
- notes: recursive = two-phase warm prior vs cold rewrite

In [ ]:
# Variant — UC6 guarded trace-feedback decision policy.
# This deterministic policy surface uses the same GuardedDecisionEvaluator to
# choose trace_type / credit_horizon from diagnostics before expensive config runs.
from opto.features.recursive_opt import GuardedDecisionCase, GuardedDecisionEvaluator


def _baseline_trace_feedback_policy(self, diagnostics):
    return "action: continue\ntrace_type: internal\ncredit_horizon: episode\nreason: default"


TRACE_FEEDBACK_CASES = (
    GuardedDecisionCase(
        name="qasper_noisy_long_context",
        payload={"task": "hf:qasper", "spread": 0.08, "wall_s": 42.0, "long_context": True},
        allowed_actions=("use", "select", "switch"),
        required_targets=("hybrid", "step"),
        required_terms=("qasper", "long", "feedback"),
        weights={"action": 0.25, "target": 0.45, "required": 0.30},
    ),
    GuardedDecisionCase(
        name="internal_fast_enough",
        payload={"task": "internal:multiobjective_gsm8k", "spread": 0.03, "wall_s": 8.0},
        allowed_actions=("use", "select", "continue"),
        required_targets=("internal", "step"),
        forbidden_targets=("otel",),
        required_terms=("fast", "direct"),
        weights={"action": 0.25, "target": 0.35, "avoid": 0.20, "required": 0.20},
        hard_forbidden=True,
    ),
    GuardedDecisionCase(
        name="otel_diagnostic_probe",
        payload={"task": "trace-debug", "missing_observability": True, "spread": 0.0},
        allowed_actions=("probe", "switch", "use"),
        required_targets=("otel", "full"),
        required_terms=("diagnostic", "observability"),
        weights={"action": 0.25, "target": 0.45, "required": 0.30},
    ),
)


def evaluate_trace_feedback_policy(component, _task_id):
    return GuardedDecisionEvaluator(TRACE_FEEDBACK_CASES)(component, _task_id)


uc6_guarded = [("guarded trace-feedback policy", run_code_experiment(
    "trace_feedback_policy", "internal:trace_feedback_policy",
    "Rewrite the trace-feedback policy. Return action, trace_type, credit_horizon, reason.",
    seeds=DIAGNOSTIC_SEEDS, memory_name="mem_uc6_guarded_trace_feedback",
    baseline=_baseline_trace_feedback_policy, evaluate=evaluate_trace_feedback_policy,
))]
show_table("Use Case 6 variant — guarded trace-feedback policy", uc6_guarded)


In [ ]:
# Variant — UC8 guarded campaign policy.
# Same campaign cases as UC8, but scored through the generic guarded-decision
# evaluator with hard forbidden-target guards for saturated/stalled cases.
from opto.features.recursive_opt import GuardedDecisionCase, GuardedDecisionEvaluator


def _campaign_guard_case(case):
    return GuardedDecisionCase(
        name=case["name"],
        payload=case["diagnostics"],
        allowed_actions=tuple(sorted(case["actions"])),
        required_targets=tuple(sorted(case["tasks"])),
        forbidden_targets=tuple(sorted(case["avoid"])),
        required_terms=tuple(sorted(case["reasons"])),
        numeric_ranges={"max_examples": case["max_examples"]},
        weights={"action": 0.30, "target": 0.25, "avoid": 0.20, "numeric": 0.15, "required": 0.10},
        required_denominator=2,
        hard_forbidden=bool(case["avoid"]),
        forbidden_floor=0.0,
    )


CAMPAIGN_GUARDED_EVALUATOR = GuardedDecisionEvaluator(
    tuple(_campaign_guard_case(case) for case in CAMPAIGN_POLICY_CASES)
)


def evaluate_campaign_policy_guarded(component, _task_id):
    return CAMPAIGN_GUARDED_EVALUATOR(component, _task_id)


uc8_guarded = [("guarded campaign policy", run_code_experiment(
    "campaign_policy_guarded", "internal:campaign_policy_guarded",
    "Rewrite the campaign policy. Return action/task/max_examples/reason and obey hard avoid guards.",
    seeds=DIAGNOSTIC_SEEDS, memory_name="mem_uc8_guarded_campaign",
    baseline=_baseline_campaign_policy, evaluate=evaluate_campaign_policy_guarded,
))]
show_table("Use Case 8 variant — guarded campaign policy", uc8_guarded)


In [ ]:
# Variant — UC11 guarded prompt-format preflight.
# This is a cheap guard layer before live QASPER scoring: it rewards prompt
# emitters that explicitly request exact/verbatim answer formats and penalizes
# paraphrase-heavy generic scholarly QA instructions.
from opto.features.recursive_opt import GuardedDecisionCase, GuardedDecisionEvaluator


QASPER_FORMAT_CASES = (
    GuardedDecisionCase(
        name="dataset_verbatim_choice",
        payload=None,
        required_terms=("exact", "dataset", "nothing else", "Chinese dataset BIBREF0"),
        forbidden_terms=("paraphrase", "broader dataset"),
        weights={"required": 0.75, "forbidden": 0.25},
        required_denominator=3,
        hard_forbidden=True,
    ),
    GuardedDecisionCase(
        name="metric_delta_exactness",
        payload=None,
        required_terms=("exact", "MRR", "MR", "Recall@10", "output only"),
        forbidden_terms=("summarize", "generic"),
        weights={"required": 0.75, "forbidden": 0.25},
        required_denominator=3,
        hard_forbidden=True,
    ),
    GuardedDecisionCase(
        name="no_wrapper_format",
        payload=None,
        required_terms=("no markdown", "no preamble", "final answer"),
        forbidden_terms=("json object", "explanation", "analysis"),
        weights={"required": 0.70, "forbidden": 0.30},
        required_denominator=2,
        hard_forbidden=True,
    ),
)


def evaluate_qasper_prompt_format_guard(component, _task_id):
    return GuardedDecisionEvaluator(QASPER_FORMAT_CASES)(component, _task_id)


uc11_guarded = [("guarded prompt-format preflight", run_code_experiment(
    "qasper_prompt_format_guard", "internal:qasper_prompt_format_guard",
    "Rewrite the QASPER prompt emitter. Emphasize exact/verbatim answer formats and no wrappers.",
    seeds=DIAGNOSTIC_SEEDS, memory_name="mem_uc11_guarded_prompt_format",
    baseline=_qasper_prompt_emitter, evaluate=evaluate_qasper_prompt_format_guard,
))]
show_table("Use Case 11 variant — guarded prompt-format preflight", uc11_guarded)


**Coverage:** the three-way benchmark is demonstrated on one use case per surface type —
**UC2** (config/prompt, spec arm + warm prior), **UC4** (family-policy/prior transfer, warm O2→O3),
**UC1** (code surface, two-phase warm prior via `make_code_arm`), and **UC13** (numeric vs
generative via `run_numeric_arm`). The remaining UCs convert by copying the matching pattern and
making their recursive arm structurally different (prior carry / extra level / numeric route).